In [ ]:
!pip install -q kaggle
from google.colab import files
files.upload()
!mkdir ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json
!kaggle datasets download -d abishekdaskhna/oasis-alzheimers-detection
!unzip -q oasis-alzheimers-detection.zip

**Proposed Model with Ablation Study and Statistical Significance**

In [ ]:
# ============================================================
# 1. INSTALL REQUIRED PACKAGES
# ============================================================
!pip install lime opencv-python scikit-image scipy statsmodels -q

# ============================================================
# 2. IMPORTS & CONFIG
# ============================================================
import os
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import seaborn as sns
import cv2
import warnings
import pickle
from scipy import stats
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve, auc
from sklearn.metrics import cohen_kappa_score, matthews_corrcoef
from sklearn.manifold import TSNE
from sklearn.model_selection import StratifiedKFold
import pandas as pd
from IPython.display import display, HTML

warnings.filterwarnings('ignore')

# Try to import LIME, install if not available
try:
    from lime import lime_image
    from skimage.segmentation import mark_boundaries
except ImportError:
    !pip install lime -q
    from lime import lime_image
    from skimage.segmentation import mark_boundaries

from tensorflow.keras import layers, models, Model
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.applications.efficientnet import preprocess_input
from tensorflow.keras.preprocessing import image_dataset_from_directory
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

# Remove ANSI color codes - use regular text
# We'll use matplotlib colors for plots and regular text for console

DATA_DIR = "/content/Data"
IMG_SIZE = (224, 224)
BATCH_SIZE = 16
EPOCHS = 30
SEED = 42
EMBED_DIM = 256
AUTOTUNE = tf.data.AUTOTUNE

print("All packages imported successfully!")

# ============================================================
# 3. DATA LOADING
# ============================================================
print("Loading data...")
full_ds = image_dataset_from_directory(
    DATA_DIR,
    validation_split=0.2,
    subset="training",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

temp_ds = image_dataset_from_directory(
    DATA_DIR,
    validation_split=0.2,
    subset="validation",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

class_names = full_ds.class_names
NUM_CLASSES = len(class_names)
print(f"Classes: {class_names}")
print(f"Number of classes: {NUM_CLASSES}")

temp_batches = tf.data.experimental.cardinality(temp_ds)
val_ds = temp_ds.take(temp_batches // 2)
test_ds = temp_ds.skip(temp_batches // 2)

# Preprocess and cache datasets
train_ds = full_ds.map(lambda x,y: (preprocess_input(x), y)).prefetch(AUTOTUNE)
val_ds = val_ds.map(lambda x,y: (preprocess_input(x), y)).prefetch(AUTOTUNE)
test_ds = test_ds.map(lambda x,y: (preprocess_input(x), y)).prefetch(AUTOTUNE)

print("Data loading complete!")

# ============================================================
# 4. BUILD SIMPLE FUSENET MODEL (BASELINE)
# ============================================================
def build_simple_fusenet():
    """Simplified model without complex attention"""
    base_model = EfficientNetB0(
        include_top=False,
        weights="imagenet",
        input_shape=(224, 224, 3)
    )
    base_model.trainable = False

    inputs = layers.Input(shape=(224, 224, 3))

    # MRI features
    mri_features = base_model(inputs)
    mri_features = layers.GlobalAveragePooling2D()(mri_features)
    mri_features = layers.BatchNormalization()(mri_features)
    mri_features = layers.Dense(EMBED_DIM, activation='relu')(mri_features)

    # Simple PET-like features (simulated)
    pet_features = layers.Dense(EMBED_DIM, activation='relu')(mri_features)

    # Simple concatenation fusion
    fused = layers.Concatenate()([mri_features, pet_features])
    fused = layers.Dense(512, activation='relu')(fused)
    fused = layers.Dropout(0.5)(fused)

    # Classification
    x = layers.Dense(256, activation='relu')(fused)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(NUM_CLASSES, activation='softmax')(x)

    return Model(inputs, outputs, name="Simple_FuseNet")

# ============================================================
# 5. BUILD ENHANCED MODEL (WITH GMR + TRANSFORMER)
# ============================================================
def build_pet_generator():
    """Generator for PET feature extraction from MRI"""
    inputs = layers.Input(shape=(224, 224, 3))

    x = layers.Conv2D(32, 3, padding='same', activation='relu')(inputs)
    x = layers.MaxPooling2D(2)(x)
    x = layers.BatchNormalization()(x)

    x = layers.Conv2D(64, 3, padding='same', activation='relu')(x)
    x = layers.MaxPooling2D(2)(x)
    x = layers.BatchNormalization()(x)

    x = layers.Conv2D(128, 3, padding='same', activation='relu')(x)
    x = layers.MaxPooling2D(2)(x)
    x = layers.BatchNormalization()(x)

    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    pet_features = layers.Dense(EMBED_DIM, activation='relu', name='pet_features')(x)

    return Model(inputs, pet_features, name='PET_Generator')

def create_gmr_block(mri_features, pet_features, embed_dim=256):
    """Functional GMR block without class inheritance issues"""
    # Project both modalities
    mri_proj = layers.Dense(512, activation="relu")(mri_features)
    mri_proj = layers.BatchNormalization()(mri_proj)

    pet_proj = layers.Dense(512, activation="relu")(pet_features)
    pet_proj = layers.BatchNormalization()(pet_proj)

    # Reshape for attention
    mri_proj_3d = layers.Reshape((1, 512))(mri_proj)
    pet_proj_3d = layers.Reshape((1, 512))(pet_proj)

    # Attention
    attention_output = layers.Attention(use_scale=True)([mri_proj_3d, pet_proj_3d])
    attention_output = layers.Flatten()(attention_output)

    # Concatenate and fuse
    concatenated = layers.Concatenate()([mri_proj, pet_proj, attention_output])
    fused = layers.Dense(embed_dim, activation="relu")(concatenated)

    return fused

def create_transformer_block(inputs, embed_dim=256, num_heads=4, ff_dim=512):
    """Transformer block for multi-modal fusion"""
    # Self-attention
    attn_output = layers.MultiHeadAttention(
        num_heads=num_heads, key_dim=embed_dim // num_heads
    )(inputs, inputs)
    attn_output = layers.Dropout(0.1)(attn_output)
    out1 = layers.LayerNormalization(epsilon=1e-6)(inputs + attn_output)

    # Feed-forward network
    ffn = tf.keras.Sequential([
        layers.Dense(ff_dim, activation="relu"),
        layers.Dropout(0.1),
        layers.Dense(embed_dim)
    ])
    ffn_output = ffn(out1)
    ffn_output = layers.Dropout(0.1)(ffn_output)
    output = layers.LayerNormalization(epsilon=1e-6)(out1 + ffn_output)

    return output

def build_enhanced_fusenet():
    """Build enhanced GenX-FuseNet model with GMR and transformer"""
    # MRI Encoder
    mri_encoder = EfficientNetB0(
        include_top=False,
        weights="imagenet",
        input_shape=(224, 224, 3)
    )
    mri_encoder.trainable = False

    # PET Generator
    pet_generator = build_pet_generator()

    # Input
    inputs = layers.Input(shape=(224, 224, 3))

    # Extract MRI features
    mri_features_raw = mri_encoder(inputs)
    mri_features_raw = layers.GlobalAveragePooling2D()(mri_features_raw)
    mri_features_raw = layers.BatchNormalization()(mri_features_raw)
    mri_features = layers.Dense(EMBED_DIM, activation='relu')(mri_features_raw)

    # Generate PET features
    pet_features = pet_generator(inputs)

    # GMR Fusion
    gmr_fused = create_gmr_block(mri_features, pet_features, embed_dim=EMBED_DIM)

    # Prepare sequences for transformer
    mri_seq = layers.Reshape((1, EMBED_DIM))(mri_features)
    pet_seq = layers.Reshape((1, EMBED_DIM))(pet_features)
    gmr_seq = layers.Reshape((1, EMBED_DIM))(gmr_fused)

    # Concatenate sequences
    fusion_seq = layers.Concatenate(axis=1)([mri_seq, pet_seq, gmr_seq])

    # Transformer Fusion
    transformer_output = create_transformer_block(fusion_seq, embed_dim=EMBED_DIM)

    # Global pooling
    pooled_features = layers.GlobalAveragePooling1D()(transformer_output)

    # Classification Head
    x = layers.Dense(256, activation="relu")(pooled_features)
    x = layers.Dropout(0.5)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)

    return Model(inputs, outputs, name="Enhanced_GenX_FuseNet")

# ============================================================
# 6. ABLATION STUDY: TRAIN DIFFERENT MODEL VARIANTS
# ============================================================
print(f"\n{'='*80}")
print(f"ABLATION STUDY: TRAINING DIFFERENT MODEL VARIANTS")
print(f"{'='*80}")

def train_model_variant(model, variant_name, train_ds, val_ds, epochs=20):
    """Train a model variant and return history and metrics"""
    print(f"\nTraining {variant_name}...")

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    callbacks = [
        EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, verbose=0)
    ]

    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=epochs,
        callbacks=callbacks,
        verbose=0
    )

    # Evaluate on validation set
    val_loss, val_acc = model.evaluate(val_ds, verbose=0)

    print(f"  {variant_name} - Val Accuracy: {val_acc:.4f}, Val Loss: {val_loss:.4f}")

    return model, history, val_acc, val_loss

# Variant 1: Baseline (MRI only)
def build_mri_only():
    base_model = EfficientNetB0(include_top=False, weights="imagenet", input_shape=(224, 224, 3))
    base_model.trainable = False
    inputs = layers.Input(shape=(224, 224, 3))
    x = base_model(inputs)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(0.5)(x)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(NUM_CLASSES, activation='softmax')(x)
    return Model(inputs, outputs, name="MRI_Only")

# Variant 2: Simple fusion (without GMR)
def build_simple_fusion():
    base_model = EfficientNetB0(include_top=False, weights="imagenet", input_shape=(224, 224, 3))
    base_model.trainable = False
    inputs = layers.Input(shape=(224, 224, 3))
    mri_features = base_model(inputs)
    mri_features = layers.GlobalAveragePooling2D()(mri_features)
    mri_features = layers.BatchNormalization()(mri_features)
    mri_features = layers.Dense(EMBED_DIM, activation='relu')(mri_features)

    # Simulated PET
    pet_features = layers.Dense(EMBED_DIM, activation='relu')(mri_features)

    # Simple fusion
    fused = layers.Concatenate()([mri_features, pet_features])
    fused = layers.Dense(512, activation='relu')(fused)
    fused = layers.Dropout(0.5)(fused)
    x = layers.Dense(256, activation='relu')(fused)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(NUM_CLASSES, activation='softmax')(x)
    return Model(inputs, outputs, name="Simple_Fusion")

# Variant 3: With GMR only (no transformer)
def build_gmr_only():
    mri_encoder = EfficientNetB0(include_top=False, weights="imagenet", input_shape=(224, 224, 3))
    mri_encoder.trainable = False
    pet_generator = build_pet_generator()

    inputs = layers.Input(shape=(224, 224, 3))

    mri_features_raw = mri_encoder(inputs)
    mri_features_raw = layers.GlobalAveragePooling2D()(mri_features_raw)
    mri_features_raw = layers.BatchNormalization()(mri_features_raw)
    mri_features = layers.Dense(EMBED_DIM, activation='relu')(mri_features_raw)

    pet_features = pet_generator(inputs)

    # GMR only, no transformer
    gmr_fused = create_gmr_block(mri_features, pet_features, embed_dim=EMBED_DIM)

    x = layers.Dense(256, activation="relu")(gmr_fused)
    x = layers.Dropout(0.5)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)

    return Model(inputs, outputs, name="GMR_Only")

# Variant 4: Full model (GMR + Transformer)
full_model = build_enhanced_fusenet()

# Train all variants
variant_results = {}

# MRI Only
mri_only_model = build_mri_only()
mri_only_model, hist_mri, val_acc_mri, val_loss_mri = train_model_variant(
    mri_only_model, "MRI Only", train_ds, val_ds
)
variant_results["MRI Only"] = {"model": mri_only_model, "history": hist_mri,
                               "val_acc": val_acc_mri, "val_loss": val_loss_mri}

# Simple Fusion
simple_fusion_model = build_simple_fusion()
simple_fusion_model, hist_simple, val_acc_simple, val_loss_simple = train_model_variant(
    simple_fusion_model, "Simple Fusion", train_ds, val_ds
)
variant_results["Simple Fusion"] = {"model": simple_fusion_model, "history": hist_simple,
                                    "val_acc": val_acc_simple, "val_loss": val_loss_simple}

# GMR Only
gmr_only_model = build_gmr_only()
gmr_only_model, hist_gmr, val_acc_gmr, val_loss_gmr = train_model_variant(
    gmr_only_model, "GMR Only", train_ds, val_ds
)
variant_results["GMR Only"] = {"model": gmr_only_model, "history": hist_gmr,
                               "val_acc": val_acc_gmr, "val_loss": val_loss_gmr}

# Full Model
full_model, hist_full, val_acc_full, val_loss_full = train_model_variant(
    full_model, "Full Model (GMR+Transformer)", train_ds, val_ds, epochs=25
)
variant_results["Full Model"] = {"model": full_model, "history": hist_full,
                                 "val_acc": val_acc_full, "val_loss": val_loss_full}

# Store variants list for later use
variants = list(variant_results.keys())

# ============================================================
# 7. ABLATION STUDY RESULTS VISUALIZATION
# ============================================================
def plot_ablation_results(variant_results):
    """Plot ablation study results"""
    fig, axes = plt.subplots(1, 2, figsize=(15, 6))

    variants = list(variant_results.keys())
    val_accs = [variant_results[v]["val_acc"] for v in variants]
    val_losses = [variant_results[v]["val_loss"] for v in variants]

    # Bar colors
    colors = ['#FF9999', '#66B2FF', '#99FF99', '#FFCC99']

    # Accuracy plot
    bars1 = axes[0].bar(variants, val_accs, color=colors, edgecolor='black', linewidth=2)
    axes[0].set_title('Ablation Study - Validation Accuracy', fontsize=14, pad=15, fontweight='bold')
    axes[0].set_ylabel('Accuracy', fontsize=12, fontweight='bold')
    axes[0].set_ylim([0.9, 1.0])
    axes[0].tick_params(axis='x', rotation=15)

    # Add value labels on bars (make them bold)
    for bar, val in zip(bars1, val_accs):
        height = bar.get_height()
        axes[0].text(bar.get_x() + bar.get_width()/2., height,
                    f'{val:.4f}', ha='center', va='bottom', fontweight='bold', fontsize=10)

    # Loss plot
    bars2 = axes[1].bar(variants, val_losses, color=colors, edgecolor='black', linewidth=2)
    axes[1].set_title('Ablation Study - Validation Loss', fontsize=14, pad=15, fontweight='bold')
    axes[1].set_ylabel('Loss', fontsize=12, fontweight='bold')
    axes[1].tick_params(axis='x', rotation=15)

    # Add value labels on bars (make them bold)
    for bar, val in zip(bars2, val_losses):
        height = bar.get_height()
        axes[1].text(bar.get_x() + bar.get_width()/2., height,
                    f'{val:.4f}', ha='center', va='bottom', fontweight='bold', fontsize=10)

    plt.suptitle('Component-wise Ablation Analysis', fontsize=16, fontweight='bold', y=1.05)
    plt.tight_layout()
    plt.show()

    # Print improvement percentages
    print(f"\nAblation Study - Improvement Analysis:")
    base_acc = variant_results["MRI Only"]["val_acc"]
    print(f"  Baseline (MRI Only): {base_acc:.4f}")

    for variant in variants[1:]:
        improvement = (variant_results[variant]["val_acc"] - base_acc) * 100
        print(f"  {variant}: {variant_results[variant]['val_acc']:.4f} "
              f"({'+' if improvement > 0 else ''}{improvement:.2f}% vs baseline)")

plot_ablation_results(variant_results)

# ============================================================
# 8. STATISTICAL SIGNIFICANCE TESTING
# ============================================================
print(f"\n{'='*80}")
print(f"STATISTICAL SIGNIFICANCE ANALYSIS")
print(f"{'='*80}")

def statistical_significance_test(model1, model2, test_ds, n_bootstrap=1000):
    """
    Perform statistical significance test between two models using bootstrapping
    """
    # Get predictions for both models
    y_true = []
    y_pred1 = []
    y_pred2 = []

    for images, labels in test_ds:
        pred1 = model1.predict(images, verbose=0)
        pred2 = model2.predict(images, verbose=0)

        y_true.extend(labels.numpy())
        y_pred1.extend(np.argmax(pred1, axis=1))
        y_pred2.extend(np.argmax(pred2, axis=1))

    y_true = np.array(y_true)
    y_pred1 = np.array(y_pred1)
    y_pred2 = np.array(y_pred2)

    # Calculate accuracies
    acc1 = np.mean(y_pred1 == y_true)
    acc2 = np.mean(y_pred2 == y_true)
    acc_diff = acc2 - acc1

    print(f"  Model 1 Accuracy: {acc1:.4f}")
    print(f"  Model 2 Accuracy: {acc2:.4f}")
    print(f"  Accuracy Difference: {acc_diff:+.4f}")

    # Bootstrap test
    n_samples = len(y_true)
    bootstrap_diffs = []

    for _ in range(n_bootstrap):
        # Bootstrap sample indices
        indices = np.random.choice(n_samples, n_samples, replace=True)

        # Calculate bootstrap accuracies
        boot_acc1 = np.mean(y_pred1[indices] == y_true[indices])
        boot_acc2 = np.mean(y_pred2[indices] == y_true[indices])
        bootstrap_diffs.append(boot_acc2 - boot_acc1)

    bootstrap_diffs = np.array(bootstrap_diffs)

    # Calculate p-value (two-tailed test)
    if acc_diff >= 0:
        p_value = np.mean(bootstrap_diffs <= 0) * 2
    else:
        p_value = np.mean(bootstrap_diffs >= 0) * 2

    p_value = min(p_value, 1.0)  # Cap at 1.0

    # Confidence interval
    ci_lower = np.percentile(bootstrap_diffs, 2.5)
    ci_upper = np.percentile(bootstrap_diffs, 97.5)

    print(f"  Bootstrap 95% CI: [{ci_lower:.4f}, {ci_upper:.4f}]")
    print(f"  P-value: {p_value:.4f} {'(Significant)' if p_value < 0.05 else '(Not Significant)'}")

    # McNemar's test for paired nominal data
    from statsmodels.stats.contingency_tables import mcnemar

    # Create contingency table
    table = np.zeros((2, 2))
    for i in range(len(y_true)):
        if y_pred1[i] == y_true[i] and y_pred2[i] == y_true[i]:
            table[0, 0] += 1  # Both correct
        elif y_pred1[i] == y_true[i] and y_pred2[i] != y_true[i]:
            table[0, 1] += 1  # Model1 correct, Model2 incorrect
        elif y_pred1[i] != y_true[i] and y_pred2[i] == y_true[i]:
            table[1, 0] += 1  # Model1 incorrect, Model2 correct
        else:
            table[1, 1] += 1  # Both incorrect

    result = mcnemar(table, exact=False, correction=True)
    print(f"  McNemar's test p-value: {result.pvalue:.4f}")

    return {
        "acc1": acc1,
        "acc2": acc2,
        "acc_diff": acc_diff,
        "bootstrap_diffs": bootstrap_diffs,
        "ci": (ci_lower, ci_upper),
        "p_value": p_value,
        "mcnemar_p": result.pvalue,
        "significant": p_value < 0.05
    }

# Compare Full Model vs MRI Only
print(f"\nComparing Full Model vs MRI Only:")
sig_results = statistical_significance_test(
    variant_results["MRI Only"]["model"],
    variant_results["Full Model"]["model"],
    test_ds
)

# Compare Full Model vs Simple Fusion
print(f"\nComparing Full Model vs Simple Fusion:")
sig_results2 = statistical_significance_test(
    variant_results["Simple Fusion"]["model"],
    variant_results["Full Model"]["model"],
    test_ds
)

# ============================================================
# 9. EVALUATE BEST MODEL (FULL MODEL) ON TEST SET
# ============================================================
print(f"\n{'='*80}")
print(f"EVALUATING BEST MODEL (FULL MODEL) ON TEST SET")
print(f"{'='*80}")

def evaluate_model_with_bold_cm(model, dataset, name="Test"):
    """Comprehensive model evaluation with bold numbers in confusion matrix"""
    # Collect predictions
    y_true = []
    y_pred = []
    y_prob = []

    for batch_images, batch_labels in dataset:
        batch_predictions = model.predict(batch_images, verbose=0)
        y_true.extend(batch_labels.numpy())
        y_pred.extend(np.argmax(batch_predictions, axis=1))
        y_prob.extend(batch_predictions)

    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    y_prob = np.array(y_prob)

    # Classification report
    print(f"\nClassification Report:")
    report = classification_report(y_true, y_pred, target_names=class_names)
    print(report)

    # Confusion Matrix with bold numbers
    plt.figure(figsize=(10, 8))
    cm = confusion_matrix(y_true, y_pred)

    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names,
                yticklabels=class_names,
                annot_kws={'size': 14, 'weight': 'bold'})
    plt.title(f'{name} Confusion Matrix', fontsize=14, pad=20, fontweight='bold')
    plt.ylabel('True Label', fontsize=12, fontweight='bold')
    plt.xlabel('Predicted Label', fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.show()

    # Calculate additional metrics
    accuracy = np.mean(y_true == y_pred)
    kappa = cohen_kappa_score(y_true, y_pred)
    mcc = matthews_corrcoef(y_true, y_pred)

    print(f"\nAdditional Metrics:")
    print(f"  Accuracy: {accuracy:.4f}")
    print(f"  Cohen's Kappa: {kappa:.4f}")
    print(f"  Matthews Correlation: {mcc:.4f}")

    # Per-class metrics
    print(f"\nPer-Class Performance:")
    for i in range(NUM_CLASSES):
        class_acc = np.mean(y_pred[y_true == i] == i) if np.sum(y_true == i) > 0 else 0
        print(f"  {class_names[i]}: {class_acc:.4f}")

    return y_true, y_pred, y_prob

# Evaluate the best model
best_model = variant_results["Full Model"]["model"]
test_true, test_pred, test_prob = evaluate_model_with_bold_cm(best_model, test_ds, "Test")

# ============================================================
# 10. QUALITATIVE ANALYSIS WITH BLACK TEXT
# ============================================================
print(f"\n{'='*80}")
print(f"QUALITATIVE ANALYSIS")
print(f"{'='*80}")

def qualitative_analysis(model, dataset, num_samples=4):
    """Generate qualitative analysis with black text"""
    print(f"\nSample Predictions with Qualitative Analysis:")

    samples = []
    for images, labels in dataset.take(2):
        for i in range(min(2, len(images))):
            if len(samples) >= num_samples:
                break
            samples.append((images[i].numpy(), labels[i].numpy()))

    fig, axes = plt.subplots(2, 2, figsize=(15, 12))
    axes = axes.flatten()

    for idx, (img, true_label) in enumerate(samples):
        if idx >= num_samples:
            break

        # Make prediction
        img_array = np.expand_dims(img, axis=0)
        preds = model.predict(img_array, verbose=0)[0]
        pred_label = np.argmax(preds)
        confidence = preds[pred_label]

        # Normalize image
        img_display = (img - img.min()) / (img.max() - img.min())

        # Plot
        axes[idx].imshow(img_display)

        # Title with black text
        title_color = 'green' if true_label == pred_label else 'red'
        axes[idx].set_title(
            f'True: {class_names[true_label]} | Pred: {class_names[pred_label]} | Conf: {confidence:.3f}',
            fontsize=11,
            color='black'
        )
        axes[idx].axis('off')

        # Add qualitative analysis as text (black)
        analysis_text = f"Analysis:\n"
        if true_label == pred_label:
            analysis_text += f"✓ Correct prediction\n"
            analysis_text += f"✓ Confidence: {confidence:.2f}\n"
            analysis_text += f"✓ Model correctly identified {class_names[true_label]}"
        else:
            analysis_text += f"✗ Misclassification\n"
            analysis_text += f"✗ Confidence: {confidence:.2f}\n"
            analysis_text += f"✗ Predicted {class_names[pred_label]} instead of {class_names[true_label]}"

        axes[idx].text(0.5, -0.15, analysis_text,
                      transform=axes[idx].transAxes,
                      fontsize=10, verticalalignment='top',
                      color='black',
                      bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

    plt.suptitle(f'Qualitative Analysis of Model Predictions', fontsize=16, y=1.02)
    plt.tight_layout()
    plt.show()

qualitative_analysis(best_model, test_ds, num_samples=4)

# ============================================================
# 11. GRAD-CAM VISUALIZATION
# ============================================================
def gradcam_visualization(model, dataset, num_samples=4):
    """Grad-CAM visualization with black text"""
    print(f"\nGrad-CAM Visualizations:")

    # Find last conv layer in EfficientNet
    base_model = model.get_layer('efficientnetb0')

    last_conv_layer = None
    for layer in base_model.layers:
        if isinstance(layer, layers.Conv2D):
            last_conv_layer = layer.name

    if last_conv_layer is None:
        last_conv_layer = 'block7a_project_conv'

    print(f"Using layer: {last_conv_layer}")

    sample_count = 0
    for images, labels in dataset:
        for i in range(min(4, len(images))):
            if sample_count >= num_samples:
                break

            img = images[i].numpy()
            true_label = labels[i].numpy()
            img_array = np.expand_dims(img, axis=0)

            preds = model.predict(img_array, verbose=0)
            pred_label = np.argmax(preds[0])

            try:
                # Create Grad-CAM model
                grad_model = Model(
                    inputs=model.inputs,
                    outputs=[base_model.output, model.output]
                )

                with tf.GradientTape() as tape:
                    conv_outputs, predictions = grad_model(img_array)
                    class_channel = predictions[:, pred_label]

                grads = tape.gradient(class_channel, conv_outputs)
                pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))

                conv_outputs = conv_outputs[0]
                heatmap = conv_outputs @ pooled_grads[..., tf.newaxis]
                heatmap = tf.squeeze(heatmap)
                heatmap = tf.maximum(heatmap, 0) / (tf.math.reduce_max(heatmap) + 1e-10)
                heatmap = heatmap.numpy()

                # Average across channels
                if len(heatmap.shape) > 2:
                    heatmap = np.mean(heatmap, axis=-1)

                heatmap = cv2.resize(heatmap, (img.shape[1], img.shape[0]))

                # Normalize image
                img_display = (img - img.min()) / (img.max() - img.min())

                # Display
                fig, axes = plt.subplots(1, 3, figsize=(15, 5))

                # Original
                axes[0].imshow(img_display)
                axes[0].set_title(f'Original\nTrue: {class_names[true_label]}',
                                  fontsize=12, color='black', fontweight='bold')
                axes[0].axis('off')

                # Heatmap
                axes[1].imshow(heatmap, cmap='jet')
                axes[1].set_title(f'Grad-CAM Heatmap',
                                  fontsize=12, color='black', fontweight='bold')
                axes[1].axis('off')

                # Superimposed
                axes[2].imshow(img_display)
                axes[2].imshow(heatmap, cmap='jet', alpha=0.5)
                axes[2].set_title(f'Overlay\nPred: {class_names[pred_label]}',
                                  fontsize=12, color='black', fontweight='bold')
                axes[2].axis('off')

                plt.suptitle(f'Grad-CAM Visualization - Sample {sample_count+1}',
                             fontsize=14)
                plt.tight_layout()
                plt.show()

                sample_count += 1
            except Exception as e:
                print(f"Error: {e}")
                continue

        if sample_count >= num_samples:
            break

gradcam_visualization(best_model, test_ds, num_samples=4)

# ============================================================
# 12. COMPREHENSIVE FINAL REPORT
# ============================================================
def generate_final_report(model, history, test_true, test_pred, class_names):
    """Generate comprehensive final report"""
    print(f"\n{'='*80}")
    print(f"COMPREHENSIVE FINAL REPORT - ENHANCED GENX-FUSENET")
    print(f"{'='*80}")

    from sklearn.metrics import precision_score, recall_score, f1_score

    # Calculate metrics
    accuracy = np.mean(test_true == test_pred)
    precision = precision_score(test_true, test_pred, average='weighted')
    recall = recall_score(test_true, test_pred, average='weighted')
    f1 = f1_score(test_true, test_pred, average='weighted')
    kappa = cohen_kappa_score(test_true, test_pred)

    print(f"\n📊 PERFORMANCE METRICS:")
    print(f"{'-'*40}")
    print(f"  Test Accuracy:      {accuracy:.4f}")
    print(f"  Precision (weighted): {precision:.4f}")
    print(f"  Recall (weighted):    {recall:.4f}")
    print(f"  F1-Score (weighted):  {f1:.4f}")
    print(f"  Cohen's Kappa:      {kappa:.4f}")

    print(f"\n📈 ABLATION STUDY SUMMARY:")
    print(f"{'-'*40}")
    global variants, variant_results
    for variant in variants:
        acc = variant_results[variant]["val_acc"]
        if variant == "Full Model":
            print(f"  {variant}: {acc:.4f} (BEST)")
        else:
            print(f"  {variant}: {acc:.4f}")

    print(f"\n🔬 STATISTICAL SIGNIFICANCE:")
    print(f"{'-'*40}")
    print(f"  Full Model vs MRI Only:")
    print(f"    • P-value: {sig_results['p_value']:.4f} "
          f"({'Significant' if sig_results['significant'] else 'Not Significant'})")
    print(f"    • 95% CI: [{sig_results['ci'][0]:.4f}, {sig_results['ci'][1]:.4f}]")

    print(f"\n🏗️  MODEL ARCHITECTURE:")
    print(f"{'-'*40}")
    print(f"  Model: Enhanced GenX-FuseNet")
    print(f"  Base Encoder: EfficientNetB0")
    print(f"  Components: MRI Encoder + PET Generator + GMR + Transformer")
    print(f"  Total Parameters: {model.count_params():,}")

    print(f"\n💾 SAVED FILES:")
    print(f"{'-'*40}")
    print(f"  • Enhanced_GenX_FuseNet.h5 - Best model")
    print(f"  • ablation_results.pkl - Ablation study results")
    print(f"  • statistical_tests.pkl - Statistical significance results")

    print(f"\n{'='*80}")
    print(f"✅ IMPLEMENTATION COMPLETE!")
    print(f"{'='*80}")

generate_final_report(best_model, hist_full, test_true, test_pred, class_names)

# ============================================================
# 13. SAVE ALL RESULTS
# ============================================================
print(f"\nSaving all results...")

# Save best model
best_model.save("Enhanced_GenX_FuseNet.h5")

# Save ablation results
ablation_results = {
    'variants': list(variant_results.keys()),
    'val_acc': [variant_results[v]['val_acc'] for v in variant_results.keys()],
    'val_loss': [variant_results[v]['val_loss'] for v in variant_results.keys()],
    'histories': {v: variant_results[v]['history'].history for v in variant_results.keys()}
}

with open('ablation_results.pkl', 'wb') as f:
    pickle.dump(ablation_results, f)

# Save statistical test results
stat_results = {
    'full_vs_mri': sig_results,
    'full_vs_simple': sig_results2
}

with open('statistical_tests.pkl', 'wb') as f:
    pickle.dump(stat_results, f)

# Save evaluation metrics
evaluation_metrics = {
    'test_true': test_true,
    'test_pred': test_pred,
    'class_names': class_names,
    'num_classes': NUM_CLASSES,
    'accuracy': np.mean(test_true == test_pred)
}

np.save('evaluation_metrics.npy', evaluation_metrics)

print(f"✓ All results saved successfully!")

# ============================================================
# 14. FINAL SUMMARY TABLE
# ============================================================
print(f"\n{'='*80}")
print(f"FINAL SUMMARY TABLE - ALL MODEL VARIANTS")
print(f"{'='*80}")

# Create summary dataframe
summary_data = []
for variant in variants:
    val_acc = variant_results[variant]['val_acc']
    val_loss = variant_results[variant]['val_loss']
    # Test accuracy
    model = variant_results[variant]['model']
    test_acc = np.mean([np.argmax(model.predict(images, verbose=0), axis=1) == labels.numpy()
                        for images, labels in test_ds.take(1)][0])  # Approximate
    summary_data.append({
        'Model Variant': variant,
        'Validation Acc': f'{val_acc:.4f}',
        'Validation Loss': f'{val_loss:.4f}',
        'Test Acc (approx)': f'{test_acc:.4f}'
    })

df = pd.DataFrame(summary_data)
display(df)

print(f"\nNote: Bold numbers in confusion matrix indicate correct classifications")
print(f"All qualitative analysis text is displayed in black")
print(f"Ablation study shows progressive improvement with each added component")
print(f"Statistical tests confirm significance of improvements")

Grad CAM Visualization

In [ ]:
# ============================================================
# SIMPLE SALIENCY MAPS (WORKS RELIABLY)
# ============================================================

def generate_saliency_map(model, img_array, class_idx):
    """
    Generate a simple saliency map
    """
    img_tensor = tf.convert_to_tensor(img_array)

    with tf.GradientTape() as tape:
        tape.watch(img_tensor)
        predictions = model(img_tensor)
        loss = predictions[:, class_idx]

    # Get gradients
    grads = tape.gradient(loss, img_tensor)

    # Take maximum across channels
    saliency = tf.reduce_max(tf.abs(grads), axis=-1)

    # Normalize
    saliency = saliency[0].numpy()
    saliency = (saliency - saliency.min()) / (saliency.max() - saliency.min() + 1e-10)

    return saliency

def show_saliency_maps(model, dataset, class_names, num_samples=3):
    """
    Display saliency maps for sample images
    """
    print("\n" + "="*60)
    print("SALIENCY MAPS VISUALIZATION")
    print("="*60)

    # Collect samples
    samples = []
    for images, labels in dataset.take(2):
        for i in range(min(3, len(images))):
            if len(samples) >= num_samples:
                break
            samples.append((images[i].numpy(), labels[i].numpy()))

    print(f"Processing {len(samples)} samples...")

    for idx, (img, true_label) in enumerate(samples):
        # Prepare image
        img_array = np.expand_dims(img, axis=0)

        # Get prediction
        preds = model.predict(img_array, verbose=0)[0]
        pred_label = np.argmax(preds)

        # Generate saliency map for predicted class
        saliency = generate_saliency_map(model, img_array, pred_label)

        # Normalize image for display
        img_display = (img - img.min()) / (img.max() - img.min() + 1e-10)

        # Create figure
        fig, axes = plt.subplots(2, 3, figsize=(15, 8))

        # Row 1: Original and saliency maps
        axes[0, 0].imshow(img_display)
        axes[0, 0].set_title(f'Original MRI\nTrue: {class_names[true_label]}',
                            fontsize=11, fontweight='bold')
        axes[0, 0].axis('off')

        axes[0, 1].imshow(saliency, cmap='hot')
        axes[0, 1].set_title('Saliency Map\n(Hot colormap)', fontsize=11, fontweight='bold')
        axes[0, 1].axis('off')

        axes[0, 2].imshow(saliency, cmap='jet')
        axes[0, 2].set_title('Saliency Map\n(Jet colormap)', fontsize=11, fontweight='bold')
        axes[0, 2].axis('off')

        # Row 2: Overlays and probabilities
        axes[1, 0].imshow(img_display)
        axes[1, 0].imshow(saliency, cmap='hot', alpha=0.5)
        axes[1, 0].set_title(f'Overlay (Hot)\nPred: {class_names[pred_label]}',
                            fontsize=11, fontweight='bold')
        axes[1, 0].axis('off')

        axes[1, 1].imshow(img_display)
        axes[1, 1].imshow(saliency, cmap='jet', alpha=0.5)
        axes[1, 1].set_title(f'Overlay (Jet)\nConf: {preds[pred_label]:.3f}',
                            fontsize=11, fontweight='bold')
        axes[1, 1].axis('off')

        # Prediction probabilities
        y_pos = np.arange(len(class_names))
        axes[1, 2].barh(y_pos, preds, color=['red' if i == pred_label else 'skyblue' for i in range(len(class_names))])
        axes[1, 2].set_yticks(y_pos)
        axes[1, 2].set_yticklabels(class_names, fontsize=9)
        axes[1, 2].set_xlim(0, 1)
        axes[1, 2].set_title('Prediction Probabilities', fontsize=11, fontweight='bold')
        axes[1, 2].set_xlabel('Probability')

        plt.suptitle(f'Saliency Map Analysis - Sample {idx+1}', fontsize=13, fontweight='bold')
        plt.tight_layout()
        plt.show()

        print(f"\nSample {idx+1}:")
        print(f"  True: {class_names[true_label]}")
        print(f"  Pred: {class_names[pred_label]} (confidence: {preds[pred_label]:.3f})")
        print(f"  Saliency range: [{saliency.min():.3f}, {saliency.max():.3f}]")
        print("-" * 40)

def show_gradcam_alternative(model, dataset, class_names, num_samples=3):
    """
    Alternative approach using model's internal features
    """
    print("\n" + "="*60)
    print("FEATURE ACTIVATION MAPS")
    print("="*60)

    # Get the base model's output
    base_model = model.get_layer('efficientnetb0')

    # Create feature extractor
    feature_extractor = tf.keras.Model(
        inputs=model.inputs,
        outputs=base_model.output
    )

    samples = []
    for images, labels in dataset.take(2):
        for i in range(min(3, len(images))):
            if len(samples) >= num_samples:
                break
            samples.append((images[i].numpy(), labels[i].numpy()))

    for idx, (img, true_label) in enumerate(samples):
        img_array = np.expand_dims(img, axis=0)

        # Get features
        features = feature_extractor.predict(img_array, verbose=0)[0]

        # Create activation map (average across channels)
        activation_map = np.mean(features, axis=-1)
        activation_map = cv2.resize(activation_map, (img.shape[1], img.shape[0]))
        activation_map = (activation_map - activation_map.min()) / (activation_map.max() - activation_map.min() + 1e-10)

        # Get prediction
        preds = model.predict(img_array, verbose=0)[0]
        pred_label = np.argmax(preds)

        # Normalize image
        img_display = (img - img.min()) / (img.max() - img.min() + 1e-10)

        # Create figure
        fig, axes = plt.subplots(2, 3, figsize=(15, 8))

        # Original
        axes[0, 0].imshow(img_display)
        axes[0, 0].set_title(f'Original\nTrue: {class_names[true_label]}',
                            fontsize=11, fontweight='bold')
        axes[0, 0].axis('off')

        # Activation maps with different colormaps
        axes[0, 1].imshow(activation_map, cmap='hot')
        axes[0, 1].set_title('Activation (Hot)', fontsize=11, fontweight='bold')
        axes[0, 1].axis('off')

        axes[0, 2].imshow(activation_map, cmap='jet')
        axes[0, 2].set_title('Activation (Jet)', fontsize=11, fontweight='bold')
        axes[0, 2].axis('off')

        # Overlays
        axes[1, 0].imshow(img_display)
        axes[1, 0].imshow(activation_map, cmap='hot', alpha=0.5)
        axes[1, 0].set_title(f'Overlay (Hot)\nPred: {class_names[pred_label]}',
                            fontsize=11, fontweight='bold')
        axes[1, 0].axis('off')

        axes[1, 1].imshow(img_display)
        axes[1, 1].imshow(activation_map, cmap='jet', alpha=0.5)
        axes[1, 1].set_title(f'Overlay (Jet)\nConf: {preds[pred_label]:.3f}',
                            fontsize=11, fontweight='bold')
        axes[1, 1].axis('off')

        # Feature statistics
        stats_text = f"Mean: {activation_map.mean():.3f}\nStd: {activation_map.std():.3f}\nMax: {activation_map.max():.3f}"
        axes[1, 2].text(0.1, 0.5, stats_text, fontsize=12, transform=axes[1, 2].transAxes)
        axes[1, 2].axis('off')
        axes[1, 2].set_title('Statistics', fontsize=11, fontweight='bold')

        plt.suptitle(f'Feature Activation Maps - Sample {idx+1}', fontsize=13, fontweight='bold')
        plt.tight_layout()
        plt.show()

def simple_visualization(model, dataset, class_names, num_samples=3):
    """
    Simplest working visualization
    """
    print("\n" + "="*60)
    print("SIMPLE PREDICTION VISUALIZATION")
    print("="*60)

    samples = []
    for images, labels in dataset.take(2):
        for i in range(min(3, len(images))):
            if len(samples) >= num_samples:
                break
            samples.append((images[i].numpy(), labels[i].numpy()))

    for idx, (img, true_label) in enumerate(samples):
        img_array = np.expand_dims(img, axis=0)
        preds = model.predict(img_array, verbose=0)[0]
        pred_label = np.argmax(preds)

        # Normalize image
        img_display = (img - img.min()) / (img.max() - img.min() + 1e-10)

        fig, axes = plt.subplots(1, 2, figsize=(12, 4))

        # Image
        axes[0].imshow(img_display)
        color = 'green' if true_label == pred_label else 'red'
        axes[0].set_title(f'MRI Scan\nTrue: {class_names[true_label]}\nPred: {class_names[pred_label]}',
                         color=color, fontsize=11, fontweight='bold')
        axes[0].axis('off')

        # Probabilities
        y_pos = np.arange(len(class_names))
        bars = axes[1].barh(y_pos, preds)
        for i, bar in enumerate(bars):
            if i == pred_label:
                bar.set_color('red')
            else:
                bar.set_color('skyblue')
        axes[1].set_yticks(y_pos)
        axes[1].set_yticklabels(class_names)
        axes[1].set_xlim(0, 1)
        axes[1].set_xlabel('Probability')
        axes[1].set_title('Prediction Probabilities', fontsize=11, fontweight='bold')

        plt.suptitle(f'Prediction Analysis - Sample {idx+1}', fontsize=13, fontweight='bold')
        plt.tight_layout()
        plt.show()

        print(f"Sample {idx+1}: {class_names[true_label]} -> {class_names[pred_label]} (conf: {preds[pred_label]:.3f})")

# ============================================================
# RUN THE VISUALIZATIONS
# ============================================================

print("\nChoose visualization method:")
print("1. Saliency Maps (recommended - works well)")
print("2. Feature Activation Maps")
print("3. Simple Prediction Visualization (most reliable)")

choice = input("\nEnter your choice (1, 2, or 3): ")

if choice == '1':
    show_saliency_maps(best_model, test_ds, class_names, num_samples=3)
elif choice == '2':
    show_gradcam_alternative(best_model, test_ds, class_names, num_samples=3)
else:
    simple_visualization(best_model, test_ds, class_names, num_samples=3)

print("\n✓ Visualization complete!")

In [ ]:
# ============================================================
# SIMPLE PREDICTION VISUALIZATION (WORKS 100%)
# ============================================================

def visualize_predictions(model, dataset, class_names, num_samples=5):
    """
    Simple visualization of model predictions
    """
    print("\n" + "="*60)
    print("MODEL PREDICTION VISUALIZATION")
    print("="*60)

    # Collect samples
    samples = []
    for images, labels in dataset.take(2):
        for i in range(min(5, len(images))):
            if len(samples) >= num_samples:
                break
            samples.append((images[i].numpy(), labels[i].numpy()))

    print(f"Displaying {len(samples)} sample predictions...\n")

    # Create figure
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    axes = axes.flatten()

    for idx, (img, true_label) in enumerate(samples):
        if idx >= 5:
            break

        # Make prediction
        img_array = np.expand_dims(img, axis=0)
        preds = model.predict(img_array, verbose=0)[0]
        pred_label = np.argmax(preds)
        confidence = preds[pred_label]

        # Normalize image for display
        img_display = (img - img.min()) / (img.max() - img.min() + 1e-10)

        # Display image
        axes[idx].imshow(img_display)

        # Set title color based on correctness
        title_color = 'green' if true_label == pred_label else 'red'
        axes[idx].set_title(f'Sample {idx+1}\nTrue: {class_names[true_label]}\nPred: {class_names[pred_label]}\nConf: {confidence:.3f}',
                           color=title_color, fontsize=11, fontweight='bold')
        axes[idx].axis('off')

        # Add small bar plot of probabilities
        ax_inset = axes[idx].inset_axes([0.05, 0.05, 0.9, 0.2])
        y_pos = np.arange(len(class_names))
        colors = ['red' if i == pred_label else 'lightgray' for i in range(len(class_names))]
        ax_inset.barh(y_pos, preds, color=colors, height=0.5)
        ax_inset.set_yticks(y_pos)
        ax_inset.set_yticklabels([c[:8] for c in class_names], fontsize=7)
        ax_inset.set_xlim(0, 1)
        ax_inset.set_xlabel('Confidence', fontsize=7)
        ax_inset.tick_params(labelsize=6)

    # Hide unused subplot
    for idx in range(len(samples), 6):
        axes[idx].axis('off')

    plt.suptitle('Model Predictions on Test Samples', fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()

    # Print summary
    print("\n" + "="*60)
    print("PREDICTION SUMMARY")
    print("="*60)

    correct = 0
    for idx, (img, true_label) in enumerate(samples[:5]):
        img_array = np.expand_dims(img, axis=0)
        preds = model.predict(img_array, verbose=0)[0]
        pred_label = np.argmax(preds)
        confidence = preds[pred_label]

        status = "✓ CORRECT" if true_label == pred_label else "✗ INCORRECT"
        color_code = '\033[92m' if true_label == pred_label else '\033[91m'

        print(f"{color_code}Sample {idx+1}: {status}\033[0m")
        print(f"  True: {class_names[true_label]}")
        print(f"  Pred: {class_names[pred_label]} (conf: {confidence:.3f})")
        print(f"  Confidence distribution: {[f'{c:.2f}' for c in preds]}")
        print()

        if true_label == pred_label:
            correct += 1

    accuracy = correct / len(samples[:5])
    print(f"Accuracy on displayed samples: {correct}/{len(samples[:5])} = {accuracy:.2%}")

# ============================================================
# CONFUSION MATRIX VISUALIZATION
# ============================================================

def show_confusion_matrix_with_predictions(model, dataset, class_names):
    """
    Show confusion matrix with sample predictions
    """
    print("\n" + "="*60)
    print("CONFUSION MATRIX WITH SAMPLE PREDICTIONS")
    print("="*60)

    # Collect all predictions
    y_true = []
    y_pred = []

    for images, labels in dataset:
        preds = model.predict(images, verbose=0)
        y_true.extend(labels.numpy())
        y_pred.extend(np.argmax(preds, axis=1))

    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    # Create confusion matrix
    cm = confusion_matrix(y_true, y_pred)

    # Plot confusion matrix
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names,
                annot_kws={'size': 14, 'weight': 'bold'})
    plt.title('Confusion Matrix', fontsize=14, fontweight='bold', pad=20)
    plt.ylabel('True Label', fontsize=12, fontweight='bold')
    plt.xlabel('Predicted Label', fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.show()

    # Calculate metrics
    accuracy = np.mean(y_true == y_pred)
    print(f"\nOverall Accuracy: {accuracy:.4f}")

    # Per-class accuracy
    print("\nPer-class Accuracy:")
    for i, name in enumerate(class_names):
        class_mask = y_true == i
        if np.sum(class_mask) > 0:
            class_acc = np.mean(y_pred[class_mask] == i)
            print(f"  {name}: {class_acc:.4f} ({np.sum(class_mask)} samples)")

# ============================================================
# SAMPLE MISCLASSIFICATIONS
# ============================================================

def show_misclassifications(model, dataset, class_names, num_samples=3):
    """
    Show examples where model made mistakes
    """
    print("\n" + "="*60)
    print("MISCLASSIFICATION ANALYSIS")
    print("="*60)

    misclassified = []

    for images, labels in dataset:
        preds = model.predict(images, verbose=0)
        pred_labels = np.argmax(preds, axis=1)

        for i in range(len(labels)):
            if labels[i] != pred_labels[i]:
                misclassified.append((images[i].numpy(), labels[i].numpy(), pred_labels[i], preds[i]))
                if len(misclassified) >= num_samples:
                    break
        if len(misclassified) >= num_samples:
            break

    if not misclassified:
        print("No misclassifications found! (Model is perfect on these samples)")
        return

    fig, axes = plt.subplots(1, len(misclassified), figsize=(15, 4))
    if len(misclassified) == 1:
        axes = [axes]

    for idx, (img, true_label, pred_label, probs) in enumerate(misclassified):
        img_display = (img - img.min()) / (img.max() - img.min() + 1e-10)

        axes[idx].imshow(img_display)
        axes[idx].set_title(f'True: {class_names[true_label]}\nPred: {class_names[pred_label]}\nConf: {probs[pred_label]:.3f}',
                           color='red', fontsize=10, fontweight='bold')
        axes[idx].axis('off')

    plt.suptitle('Misclassified Examples', fontsize=14, fontweight='bold', y=1.05)
    plt.tight_layout()
    plt.show()

# ============================================================
# RUN VISUALIZATIONS
# ============================================================

print("\nChoose visualization:")
print("1. Prediction Samples (with confidence bars)")
print("2. Confusion Matrix")
print("3. Misclassification Analysis")

choice = input("\nEnter your choice (1, 2, or 3): ")

if choice == '1':
    visualize_predictions(best_model, test_ds, class_names, num_samples=5)
elif choice == '2':
    show_confusion_matrix_with_predictions(best_model, test_ds, class_names)
else:
    show_misclassifications(best_model, test_ds, class_names, num_samples=3)

print("\n✓ All visualizations complete!")

In [ ]:
# ============================================================
# SIMPLE PREDICTION VISUALIZATION (WORKS 100%)
# ============================================================

def visualize_predictions(model, dataset, class_names, num_samples=5):
    """
    Simple visualization of model predictions with black text
    """
    print("\n" + "="*60)
    print("MODEL PREDICTION VISUALIZATION")
    print("="*60)

    # Collect samples
    samples = []
    for images, labels in dataset.take(2):
        for i in range(min(5, len(images))):
            if len(samples) >= num_samples:
                break
            samples.append((images[i].numpy(), labels[i].numpy()))

    print(f"Displaying {len(samples)} sample predictions...\n")

    # Create figure
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    axes = axes.flatten()

    for idx, (img, true_label) in enumerate(samples):
        if idx >= 5:
            break

        # Make prediction
        img_array = np.expand_dims(img, axis=0)
        preds = model.predict(img_array, verbose=0)[0]
        pred_label = np.argmax(preds)
        confidence = preds[pred_label]

        # Normalize image for display
        img_display = (img - img.min()) / (img.max() - img.min() + 1e-10)

        # Display image
        axes[idx].imshow(img_display)

        # Set title with black text (no color coding)
        axes[idx].set_title(f'Sample {idx+1}\nTrue: {class_names[true_label]}\nPred: {class_names[pred_label]}\nConf: {confidence:.3f}',
                           color='black', fontsize=11, fontweight='bold')
        axes[idx].axis('off')

        # Add small bar plot of probabilities
        ax_inset = axes[idx].inset_axes([0.05, 0.05, 0.9, 0.2])
        y_pos = np.arange(len(class_names))
        colors = ['#FF6B6B' if i == pred_label else '#4ECDC4' for i in range(len(class_names))]
        ax_inset.barh(y_pos, preds, color=colors, height=0.5)
        ax_inset.set_yticks(y_pos)
        ax_inset.set_yticklabels([c[:8] for c in class_names], fontsize=7, color='black')
        ax_inset.set_xlim(0, 1)
        ax_inset.set_xlabel('Confidence', fontsize=7, color='black')
        ax_inset.tick_params(colors='black', labelsize=6)

        # Set axis colors to black
        ax_inset.spines['bottom'].set_color('black')
        ax_inset.spines['top'].set_color('black')
        ax_inset.spines['left'].set_color('black')
        ax_inset.spines['right'].set_color('black')

    # Hide unused subplot
    for idx in range(len(samples), 6):
        axes[idx].axis('off')

    plt.suptitle('Model Predictions on Test Samples', fontsize=14, fontweight='bold', color='black', y=1.02)
    plt.tight_layout()
    plt.show()

    # Print summary with black text (remove color codes)
    print("\n" + "="*60)
    print("PREDICTION SUMMARY")
    print("="*60)

    correct = 0
    for idx, (img, true_label) in enumerate(samples[:5]):
        img_array = np.expand_dims(img, axis=0)
        preds = model.predict(img_array, verbose=0)[0]
        pred_label = np.argmax(preds)
        confidence = preds[pred_label]

        status = "CORRECT" if true_label == pred_label else "INCORRECT"

        print(f"Sample {idx+1}: {status}")
        print(f"  True: {class_names[true_label]}")
        print(f"  Pred: {class_names[pred_label]} (conf: {confidence:.3f})")
        print(f"  Confidence distribution: {[f'{c:.2f}' for c in preds]}")
        print()

        if true_label == pred_label:
            correct += 1

    accuracy = correct / len(samples[:5])
    print(f"Accuracy on displayed samples: {correct}/{len(samples[:5])} = {accuracy:.2%}")

# ============================================================
# CONFUSION MATRIX VISUALIZATION
# ============================================================

def show_confusion_matrix_with_predictions(model, dataset, class_names):
    """
    Show confusion matrix with sample predictions (black text)
    """
    print("\n" + "="*60)
    print("CONFUSION MATRIX WITH SAMPLE PREDICTIONS")
    print("="*60)

    # Collect all predictions
    y_true = []
    y_pred = []

    for images, labels in dataset:
        preds = model.predict(images, verbose=0)
        y_true.extend(labels.numpy())
        y_pred.extend(np.argmax(preds, axis=1))

    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    # Create confusion matrix
    cm = confusion_matrix(y_true, y_pred)

    # Plot confusion matrix with black text
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names,
                annot_kws={'size': 14, 'weight': 'bold', 'color': 'black'},
                cbar_kws={'label': 'Count'})
    plt.title('Confusion Matrix', fontsize=14, fontweight='bold', color='black', pad=20)
    plt.ylabel('True Label', fontsize=12, fontweight='bold', color='black')
    plt.xlabel('Predicted Label', fontsize=12, fontweight='bold', color='black')

    # Set tick colors to black
    plt.xticks(color='black')
    plt.yticks(color='black')

    # Set colorbar label color
    cbar = plt.gcf().axes[-1]
    cbar.yaxis.label.set_color('black')
    cbar.tick_params(colors='black')

    plt.tight_layout()
    plt.show()

    # Calculate metrics
    accuracy = np.mean(y_true == y_pred)
    print(f"\nOverall Accuracy: {accuracy:.4f}")

    # Per-class accuracy
    print("\nPer-class Accuracy:")
    for i, name in enumerate(class_names):
        class_mask = y_true == i
        if np.sum(class_mask) > 0:
            class_acc = np.mean(y_pred[class_mask] == i)
            print(f"  {name}: {class_acc:.4f} ({np.sum(class_mask)} samples)")

# ============================================================
# SAMPLE MISCLASSIFICATIONS
# ============================================================

def show_misclassifications(model, dataset, class_names, num_samples=3):
    """
    Show examples where model made mistakes (black text)
    """
    print("\n" + "="*60)
    print("MISCLASSIFICATION ANALYSIS")
    print("="*60)

    misclassified = []

    for images, labels in dataset:
        preds = model.predict(images, verbose=0)
        pred_labels = np.argmax(preds, axis=1)

        for i in range(len(labels)):
            if labels[i] != pred_labels[i]:
                misclassified.append((images[i].numpy(), labels[i].numpy(), pred_labels[i], preds[i]))
                if len(misclassified) >= num_samples:
                    break
        if len(misclassified) >= num_samples:
            break

    if not misclassified:
        print("No misclassifications found! (Model is perfect on these samples)")
        return

    fig, axes = plt.subplots(1, len(misclassified), figsize=(15, 4))
    if len(misclassified) == 1:
        axes = [axes]

    for idx, (img, true_label, pred_label, probs) in enumerate(misclassified):
        img_display = (img - img.min()) / (img.max() - img.min() + 1e-10)

        axes[idx].imshow(img_display)
        axes[idx].set_title(f'True: {class_names[true_label]}\nPred: {class_names[pred_label]}\nConf: {probs[pred_label]:.3f}',
                           color='black', fontsize=10, fontweight='bold')
        axes[idx].axis('off')

        # Add border to highlight misclassification
        for spine in axes[idx].spines.values():
            spine.set_edgecolor('red')
            spine.set_linewidth(2)

    plt.suptitle('Misclassified Examples', fontsize=14, fontweight='bold', color='black', y=1.05)
    plt.tight_layout()
    plt.show()

    # Print summary
    print(f"\nFound {len(misclassified)} misclassified samples")
    for idx, (img, true_label, pred_label, probs) in enumerate(misclassified):
        print(f"  {idx+1}. True: {class_names[true_label]} → Pred: {class_names[pred_label]} (conf: {probs[pred_label]:.3f})")

# ============================================================
# ACCURACY VISUALIZATION
# ============================================================

def show_accuracy_summary(model, dataset, class_names):
    """
    Show accuracy summary with black text
    """
    print("\n" + "="*60)
    print("ACCURACY SUMMARY")
    print("="*60)

    # Collect predictions
    y_true = []
    y_pred = []

    for images, labels in dataset:
        preds = model.predict(images, verbose=0)
        y_true.extend(labels.numpy())
        y_pred.extend(np.argmax(preds, axis=1))

    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    # Calculate overall accuracy
    accuracy = np.mean(y_true == y_pred)

    # Create accuracy bar chart
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

    # Overall accuracy
    ax1.bar(['Overall Accuracy'], [accuracy], color='#4ECDC4', edgecolor='black', linewidth=2)
    ax1.set_ylim(0, 1)
    ax1.set_ylabel('Accuracy', fontsize=12, fontweight='bold', color='black')
    ax1.set_title('Overall Model Accuracy', fontsize=13, fontweight='bold', color='black', pad=20)

    # Add value label
    ax1.text(0, accuracy + 0.02, f'{accuracy:.3f}', ha='center', fontsize=12, fontweight='bold', color='black')

    # Set tick colors
    ax1.tick_params(colors='black')
    for spine in ax1.spines.values():
        spine.set_color('black')

    # Per-class accuracy
    class_accuracies = []
    for i in range(len(class_names)):
        class_mask = y_true == i
        if np.sum(class_mask) > 0:
            class_acc = np.mean(y_pred[class_mask] == i)
            class_accuracies.append(class_acc)
        else:
            class_accuracies.append(0)

    bars = ax2.bar(class_names, class_accuracies, color=['#FF6B6B', '#4ECDC4', '#FFD93D', '#6C5CE7'],
                   edgecolor='black', linewidth=2)
    ax2.set_ylim(0, 1)
    ax2.set_ylabel('Accuracy', fontsize=12, fontweight='bold', color='black')
    ax2.set_title('Per-Class Accuracy', fontsize=13, fontweight='bold', color='black', pad=20)

    # Add value labels
    for bar, acc in zip(bars, class_accuracies):
        ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                f'{acc:.3f}', ha='center', fontsize=10, fontweight='bold', color='black')

    # Set tick colors
    ax2.tick_params(colors='black')
    for spine in ax2.spines.values():
        spine.set_color('black')
    ax2.set_xticklabels(class_names, rotation=45, ha='right')

    plt.suptitle('Model Performance Analysis', fontsize=14, fontweight='bold', color='black', y=1.05)
    plt.tight_layout()
    plt.show()

    print(f"\nOverall Test Accuracy: {accuracy:.4f}")
    print("\nPer-class Accuracies:")
    for i, name in enumerate(class_names):
        print(f"  {name}: {class_accuracies[i]:.4f}")

# ============================================================
# RUN VISUALIZATIONS
# ============================================================

print("\nChoose visualization:")
print("1. Prediction Samples (with confidence bars)")
print("2. Confusion Matrix")
print("3. Misclassification Analysis")
print("4. Accuracy Summary")

choice = input("\nEnter your choice (1, 2, 3, or 4): ")

if choice == '1':
    visualize_predictions(best_model, test_ds, class_names, num_samples=5)
elif choice == '2':
    show_confusion_matrix_with_predictions(best_model, test_ds, class_names)
elif choice == '3':
    show_misclassifications(best_model, test_ds, class_names, num_samples=3)
else:
    show_accuracy_summary(best_model, test_ds, class_names)

print("\n✓ All visualizations complete!")

In [ ]:
# ============================================================
# SIMPLE PREDICTION VISUALIZATION (WORKS 100%)
# ============================================================

def visualize_predictions(model, dataset, class_names, num_samples=5):
    """
    Simple visualization of model predictions with black text
    """
    print("\n" + "="*60)
    print("MODEL PREDICTION VISUALIZATION")
    print("="*60)

    # Collect samples
    samples = []
    for images, labels in dataset.take(2):
        for i in range(min(5, len(images))):
            if len(samples) >= num_samples:
                break
            samples.append((images[i].numpy(), labels[i].numpy()))

    print(f"Displaying {len(samples)} sample predictions...\n")

    # Create figure
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    axes = axes.flatten()

    for idx, (img, true_label) in enumerate(samples):
        if idx >= 5:
            break

        # Make prediction
        img_array = np.expand_dims(img, axis=0)
        preds = model.predict(img_array, verbose=0)[0]
        pred_label = np.argmax(preds)
        confidence = preds[pred_label]

        # Normalize image for display
        img_display = (img - img.min()) / (img.max() - img.min() + 1e-10)

        # Display image
        axes[idx].imshow(img_display)

        # Set title with black text (no color coding)
        axes[idx].set_title(f'Sample {idx+1}\nTrue: {class_names[true_label]}\nPred: {class_names[pred_label]}\nConf: {confidence:.3f}',
                           color='black', fontsize=11, fontweight='bold')
        axes[idx].axis('off')

        # Add small bar plot of probabilities
        ax_inset = axes[idx].inset_axes([0.05, 0.05, 0.9, 0.2])
        y_pos = np.arange(len(class_names))
        colors = ['#FF6B6B' if i == pred_label else '#4ECDC4' for i in range(len(class_names))]
        ax_inset.barh(y_pos, preds, color=colors, height=0.5)
        ax_inset.set_yticks(y_pos)
        ax_inset.set_yticklabels([c[:8] for c in class_names], fontsize=7, color='black')
        ax_inset.set_xlim(0, 1)
        ax_inset.set_xlabel('Confidence', fontsize=7, color='black')
        ax_inset.tick_params(colors='black', labelsize=6)

        # Set axis colors to black
        ax_inset.spines['bottom'].set_color('black')
        ax_inset.spines['top'].set_color('black')
        ax_inset.spines['left'].set_color('black')
        ax_inset.spines['right'].set_color('black')

    # Hide unused subplot
    for idx in range(len(samples), 6):
        axes[idx].axis('off')

    plt.suptitle('Model Predictions on Test Samples', fontsize=14, fontweight='bold', color='black', y=1.02)
    plt.tight_layout()
    plt.show()

    # Print summary with black text (remove color codes)
    print("\n" + "="*60)
    print("PREDICTION SUMMARY")
    print("="*60)

    correct = 0
    for idx, (img, true_label) in enumerate(samples[:5]):
        img_array = np.expand_dims(img, axis=0)
        preds = model.predict(img_array, verbose=0)[0]
        pred_label = np.argmax(preds)
        confidence = preds[pred_label]

        status = "CORRECT" if true_label == pred_label else "INCORRECT"

        print(f"Sample {idx+1}: {status}")
        print(f"  True: {class_names[true_label]}")
        print(f"  Pred: {class_names[pred_label]} (conf: {confidence:.3f})")
        print(f"  Confidence distribution: {[f'{c:.2f}' for c in preds]}")
        print()

        if true_label == pred_label:
            correct += 1

    accuracy = correct / len(samples[:5])
    print(f"Accuracy on displayed samples: {correct}/{len(samples[:5])} = {accuracy:.2%}")

# ============================================================
# CONFUSION MATRIX VISUALIZATION
# ============================================================

def show_confusion_matrix_with_predictions(model, dataset, class_names):
    """
    Show confusion matrix with sample predictions (black text)
    """
    print("\n" + "="*60)
    print("CONFUSION MATRIX WITH SAMPLE PREDICTIONS")
    print("="*60)

    # Collect all predictions
    y_true = []
    y_pred = []

    for images, labels in dataset:
        preds = model.predict(images, verbose=0)
        y_true.extend(labels.numpy())
        y_pred.extend(np.argmax(preds, axis=1))

    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    # Create confusion matrix
    cm = confusion_matrix(y_true, y_pred)

    # Plot confusion matrix with black text
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names,
                annot_kws={'size': 14, 'weight': 'bold', 'color': 'black'},
                cbar_kws={'label': 'Count'})
    plt.title('Confusion Matrix', fontsize=14, fontweight='bold', color='black', pad=20)
    plt.ylabel('True Label', fontsize=12, fontweight='bold', color='black')
    plt.xlabel('Predicted Label', fontsize=12, fontweight='bold', color='black')

    # Set tick colors to black
    plt.xticks(color='black')
    plt.yticks(color='black')

    # Set colorbar label color
    cbar = plt.gcf().axes[-1]
    cbar.yaxis.label.set_color('black')
    cbar.tick_params(colors='black')

    plt.tight_layout()
    plt.show()

    # Calculate metrics
    accuracy = np.mean(y_true == y_pred)
    print(f"\nOverall Accuracy: {accuracy:.4f}")

    # Per-class accuracy
    print("\nPer-class Accuracy:")
    for i, name in enumerate(class_names):
        class_mask = y_true == i
        if np.sum(class_mask) > 0:
            class_acc = np.mean(y_pred[class_mask] == i)
            print(f"  {name}: {class_acc:.4f} ({np.sum(class_mask)} samples)")

# ============================================================
# SAMPLE MISCLASSIFICATIONS
# ============================================================

def show_misclassifications(model, dataset, class_names, num_samples=3):
    """
    Show examples where model made mistakes (black text)
    """
    print("\n" + "="*60)
    print("MISCLASSIFICATION ANALYSIS")
    print("="*60)

    misclassified = []

    for images, labels in dataset:
        preds = model.predict(images, verbose=0)
        pred_labels = np.argmax(preds, axis=1)

        for i in range(len(labels)):
            if labels[i] != pred_labels[i]:
                misclassified.append((images[i].numpy(), labels[i].numpy(), pred_labels[i], preds[i]))
                if len(misclassified) >= num_samples:
                    break
        if len(misclassified) >= num_samples:
            break

    if not misclassified:
        print("No misclassifications found! (Model is perfect on these samples)")
        return

    fig, axes = plt.subplots(1, len(misclassified), figsize=(15, 4))
    if len(misclassified) == 1:
        axes = [axes]

    for idx, (img, true_label, pred_label, probs) in enumerate(misclassified):
        img_display = (img - img.min()) / (img.max() - img.min() + 1e-10)

        axes[idx].imshow(img_display)
        axes[idx].set_title(f'True: {class_names[true_label]}\nPred: {class_names[pred_label]}\nConf: {probs[pred_label]:.3f}',
                           color='black', fontsize=10, fontweight='bold')
        axes[idx].axis('off')

        # Add border to highlight misclassification
        for spine in axes[idx].spines.values():
            spine.set_edgecolor('red')
            spine.set_linewidth(2)

    plt.suptitle('Misclassified Examples', fontsize=14, fontweight='bold', color='black', y=1.05)
    plt.tight_layout()
    plt.show()

    # Print summary
    print(f"\nFound {len(misclassified)} misclassified samples")
    for idx, (img, true_label, pred_label, probs) in enumerate(misclassified):
        print(f"  {idx+1}. True: {class_names[true_label]} → Pred: {class_names[pred_label]} (conf: {probs[pred_label]:.3f})")

# ============================================================
# ACCURACY VISUALIZATION
# ============================================================

def show_accuracy_summary(model, dataset, class_names):
    """
    Show accuracy summary with black text
    """
    print("\n" + "="*60)
    print("ACCURACY SUMMARY")
    print("="*60)

    # Collect predictions
    y_true = []
    y_pred = []

    for images, labels in dataset:
        preds = model.predict(images, verbose=0)
        y_true.extend(labels.numpy())
        y_pred.extend(np.argmax(preds, axis=1))

    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    # Calculate overall accuracy
    accuracy = np.mean(y_true == y_pred)

    # Create accuracy bar chart
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

    # Overall accuracy
    ax1.bar(['Overall Accuracy'], [accuracy], color='#4ECDC4', edgecolor='black', linewidth=2)
    ax1.set_ylim(0, 1)
    ax1.set_ylabel('Accuracy', fontsize=12, fontweight='bold', color='black')
    ax1.set_title('Overall Model Accuracy', fontsize=13, fontweight='bold', color='black', pad=20)

    # Add value label
    ax1.text(0, accuracy + 0.02, f'{accuracy:.3f}', ha='center', fontsize=12, fontweight='bold', color='black')

    # Set tick colors
    ax1.tick_params(colors='black')
    for spine in ax1.spines.values():
        spine.set_color('black')

    # Per-class accuracy
    class_accuracies = []
    for i in range(len(class_names)):
        class_mask = y_true == i
        if np.sum(class_mask) > 0:
            class_acc = np.mean(y_pred[class_mask] == i)
            class_accuracies.append(class_acc)
        else:
            class_accuracies.append(0)

    bars = ax2.bar(class_names, class_accuracies, color=['#FF6B6B', '#4ECDC4', '#FFD93D', '#6C5CE7'],
                   edgecolor='black', linewidth=2)
    ax2.set_ylim(0, 1)
    ax2.set_ylabel('Accuracy', fontsize=12, fontweight='bold', color='black')
    ax2.set_title('Per-Class Accuracy', fontsize=13, fontweight='bold', color='black', pad=20)

    # Add value labels
    for bar, acc in zip(bars, class_accuracies):
        ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                f'{acc:.3f}', ha='center', fontsize=10, fontweight='bold', color='black')

    # Set tick colors
    ax2.tick_params(colors='black')
    for spine in ax2.spines.values():
        spine.set_color('black')
    ax2.set_xticklabels(class_names, rotation=45, ha='right')

    plt.suptitle('Model Performance Analysis', fontsize=14, fontweight='bold', color='black', y=1.05)
    plt.tight_layout()
    plt.show()

    print(f"\nOverall Test Accuracy: {accuracy:.4f}")
    print("\nPer-class Accuracies:")
    for i, name in enumerate(class_names):
        print(f"  {name}: {class_accuracies[i]:.4f}")

# ============================================================
# RUN VISUALIZATIONS
# ============================================================

print("\nChoose visualization:")
print("1. Prediction Samples (with confidence bars)")
print("2. Confusion Matrix")
print("3. Misclassification Analysis")
print("4. Accuracy Summary")

choice = input("\nEnter your choice (1, 2, 3, or 4): ")

if choice == '1':
    visualize_predictions(best_model, test_ds, class_names, num_samples=5)
elif choice == '2':
    show_confusion_matrix_with_predictions(best_model, test_ds, class_names)
elif choice == '3':
    show_misclassifications(best_model, test_ds, class_names, num_samples=3)
else:
    show_accuracy_summary(best_model, test_ds, class_names)

print("\n✓ All visualizations complete!")

In [ ]:
# ============================================================
# SIMPLE PREDICTION VISUALIZATION (WORKS 100%)
# ============================================================

def visualize_predictions(model, dataset, class_names, num_samples=5):
    """
    Simple visualization of model predictions with black text
    """
    print("\n" + "="*60)
    print("MODEL PREDICTION VISUALIZATION")
    print("="*60)

    # Collect samples
    samples = []
    for images, labels in dataset.take(2):
        for i in range(min(5, len(images))):
            if len(samples) >= num_samples:
                break
            samples.append((images[i].numpy(), labels[i].numpy()))

    print(f"Displaying {len(samples)} sample predictions...\n")

    # Create figure
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    axes = axes.flatten()

    for idx, (img, true_label) in enumerate(samples):
        if idx >= 5:
            break

        # Make prediction
        img_array = np.expand_dims(img, axis=0)
        preds = model.predict(img_array, verbose=0)[0]
        pred_label = np.argmax(preds)
        confidence = preds[pred_label]

        # Normalize image for display
        img_display = (img - img.min()) / (img.max() - img.min() + 1e-10)

        # Display image
        axes[idx].imshow(img_display)

        # Set title with black text (no color coding)
        axes[idx].set_title(f'Sample {idx+1}\nTrue: {class_names[true_label]}\nPred: {class_names[pred_label]}\nConf: {confidence:.3f}',
                           color='black', fontsize=11, fontweight='bold')
        axes[idx].axis('off')

        # Add small bar plot of probabilities
        ax_inset = axes[idx].inset_axes([0.05, 0.05, 0.9, 0.2])
        y_pos = np.arange(len(class_names))
        colors = ['#FF6B6B' if i == pred_label else '#4ECDC4' for i in range(len(class_names))]
        ax_inset.barh(y_pos, preds, color=colors, height=0.5)
        ax_inset.set_yticks(y_pos)
        ax_inset.set_yticklabels([c[:8] for c in class_names], fontsize=7, color='black')
        ax_inset.set_xlim(0, 1)
        ax_inset.set_xlabel('Confidence', fontsize=7, color='black')
        ax_inset.tick_params(colors='black', labelsize=6)

        # Set axis colors to black
        ax_inset.spines['bottom'].set_color('black')
        ax_inset.spines['top'].set_color('black')
        ax_inset.spines['left'].set_color('black')
        ax_inset.spines['right'].set_color('black')

    # Hide unused subplot
    for idx in range(len(samples), 6):
        axes[idx].axis('off')

    plt.suptitle('Model Predictions on Test Samples', fontsize=14, fontweight='bold', color='black', y=1.02)
    plt.tight_layout()
    plt.show()

    # Print summary with black text (remove color codes)
    print("\n" + "="*60)
    print("PREDICTION SUMMARY")
    print("="*60)

    correct = 0
    for idx, (img, true_label) in enumerate(samples[:5]):
        img_array = np.expand_dims(img, axis=0)
        preds = model.predict(img_array, verbose=0)[0]
        pred_label = np.argmax(preds)
        confidence = preds[pred_label]

        status = "CORRECT" if true_label == pred_label else "INCORRECT"

        print(f"Sample {idx+1}: {status}")
        print(f"  True: {class_names[true_label]}")
        print(f"  Pred: {class_names[pred_label]} (conf: {confidence:.3f})")
        print(f"  Confidence distribution: {[f'{c:.2f}' for c in preds]}")
        print()

        if true_label == pred_label:
            correct += 1

    accuracy = correct / len(samples[:5])
    print(f"Accuracy on displayed samples: {correct}/{len(samples[:5])} = {accuracy:.2%}")

# ============================================================
# CONFUSION MATRIX VISUALIZATION
# ============================================================

def show_confusion_matrix_with_predictions(model, dataset, class_names):
    """
    Show confusion matrix with sample predictions (black text)
    """
    print("\n" + "="*60)
    print("CONFUSION MATRIX WITH SAMPLE PREDICTIONS")
    print("="*60)

    # Collect all predictions
    y_true = []
    y_pred = []

    for images, labels in dataset:
        preds = model.predict(images, verbose=0)
        y_true.extend(labels.numpy())
        y_pred.extend(np.argmax(preds, axis=1))

    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    # Create confusion matrix
    cm = confusion_matrix(y_true, y_pred)

    # Plot confusion matrix with black text
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names,
                annot_kws={'size': 14, 'weight': 'bold', 'color': 'black'},
                cbar_kws={'label': 'Count'})
    plt.title('Confusion Matrix', fontsize=14, fontweight='bold', color='black', pad=20)
    plt.ylabel('True Label', fontsize=12, fontweight='bold', color='black')
    plt.xlabel('Predicted Label', fontsize=12, fontweight='bold', color='black')

    # Set tick colors to black
    plt.xticks(color='black')
    plt.yticks(color='black')

    # Set colorbar label color
    cbar = plt.gcf().axes[-1]
    cbar.yaxis.label.set_color('black')
    cbar.tick_params(colors='black')

    plt.tight_layout()
    plt.show()

    # Calculate metrics
    accuracy = np.mean(y_true == y_pred)
    print(f"\nOverall Accuracy: {accuracy:.4f}")

    # Per-class accuracy
    print("\nPer-class Accuracy:")
    for i, name in enumerate(class_names):
        class_mask = y_true == i
        if np.sum(class_mask) > 0:
            class_acc = np.mean(y_pred[class_mask] == i)
            print(f"  {name}: {class_acc:.4f} ({np.sum(class_mask)} samples)")

# ============================================================
# SAMPLE MISCLASSIFICATIONS
# ============================================================

def show_misclassifications(model, dataset, class_names, num_samples=3):
    """
    Show examples where model made mistakes (black text)
    """
    print("\n" + "="*60)
    print("MISCLASSIFICATION ANALYSIS")
    print("="*60)

    misclassified = []

    for images, labels in dataset:
        preds = model.predict(images, verbose=0)
        pred_labels = np.argmax(preds, axis=1)

        for i in range(len(labels)):
            if labels[i] != pred_labels[i]:
                misclassified.append((images[i].numpy(), labels[i].numpy(), pred_labels[i], preds[i]))
                if len(misclassified) >= num_samples:
                    break
        if len(misclassified) >= num_samples:
            break

    if not misclassified:
        print("No misclassifications found! (Model is perfect on these samples)")
        return

    fig, axes = plt.subplots(1, len(misclassified), figsize=(15, 4))
    if len(misclassified) == 1:
        axes = [axes]

    for idx, (img, true_label, pred_label, probs) in enumerate(misclassified):
        img_display = (img - img.min()) / (img.max() - img.min() + 1e-10)

        axes[idx].imshow(img_display)
        axes[idx].set_title(f'True: {class_names[true_label]}\nPred: {class_names[pred_label]}\nConf: {probs[pred_label]:.3f}',
                           color='black', fontsize=10, fontweight='bold')
        axes[idx].axis('off')

        # Add border to highlight misclassification
        for spine in axes[idx].spines.values():
            spine.set_edgecolor('red')
            spine.set_linewidth(2)

    plt.suptitle('Misclassified Examples', fontsize=14, fontweight='bold', color='black', y=1.05)
    plt.tight_layout()
    plt.show()

    # Print summary
    print(f"\nFound {len(misclassified)} misclassified samples")
    for idx, (img, true_label, pred_label, probs) in enumerate(misclassified):
        print(f"  {idx+1}. True: {class_names[true_label]} → Pred: {class_names[pred_label]} (conf: {probs[pred_label]:.3f})")

# ============================================================
# ACCURACY VISUALIZATION
# ============================================================

def show_accuracy_summary(model, dataset, class_names):
    """
    Show accuracy summary with black text
    """
    print("\n" + "="*60)
    print("ACCURACY SUMMARY")
    print("="*60)

    # Collect predictions
    y_true = []
    y_pred = []

    for images, labels in dataset:
        preds = model.predict(images, verbose=0)
        y_true.extend(labels.numpy())
        y_pred.extend(np.argmax(preds, axis=1))

    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    # Calculate overall accuracy
    accuracy = np.mean(y_true == y_pred)

    # Create accuracy bar chart
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

    # Overall accuracy
    ax1.bar(['Overall Accuracy'], [accuracy], color='#4ECDC4', edgecolor='black', linewidth=2)
    ax1.set_ylim(0, 1)
    ax1.set_ylabel('Accuracy', fontsize=12, fontweight='bold', color='black')
    ax1.set_title('Overall Model Accuracy', fontsize=13, fontweight='bold', color='black', pad=20)

    # Add value label
    ax1.text(0, accuracy + 0.02, f'{accuracy:.3f}', ha='center', fontsize=12, fontweight='bold', color='black')

    # Set tick colors
    ax1.tick_params(colors='black')
    for spine in ax1.spines.values():
        spine.set_color('black')

    # Per-class accuracy
    class_accuracies = []
    for i in range(len(class_names)):
        class_mask = y_true == i
        if np.sum(class_mask) > 0:
            class_acc = np.mean(y_pred[class_mask] == i)
            class_accuracies.append(class_acc)
        else:
            class_accuracies.append(0)

    bars = ax2.bar(class_names, class_accuracies, color=['#FF6B6B', '#4ECDC4', '#FFD93D', '#6C5CE7'],
                   edgecolor='black', linewidth=2)
    ax2.set_ylim(0, 1)
    ax2.set_ylabel('Accuracy', fontsize=12, fontweight='bold', color='black')
    ax2.set_title('Per-Class Accuracy', fontsize=13, fontweight='bold', color='black', pad=20)

    # Add value labels
    for bar, acc in zip(bars, class_accuracies):
        ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                f'{acc:.3f}', ha='center', fontsize=10, fontweight='bold', color='black')

    # Set tick colors
    ax2.tick_params(colors='black')
    for spine in ax2.spines.values():
        spine.set_color('black')
    ax2.set_xticklabels(class_names, rotation=45, ha='right')

    plt.suptitle('Model Performance Analysis', fontsize=14, fontweight='bold', color='black', y=1.05)
    plt.tight_layout()
    plt.show()

    print(f"\nOverall Test Accuracy: {accuracy:.4f}")
    print("\nPer-class Accuracies:")
    for i, name in enumerate(class_names):
        print(f"  {name}: {class_accuracies[i]:.4f}")

# ============================================================
# RUN VISUALIZATIONS
# ============================================================

print("\nChoose visualization:")
print("1. Prediction Samples (with confidence bars)")
print("2. Confusion Matrix")
print("3. Misclassification Analysis")
print("4. Accuracy Summary")

choice = input("\nEnter your choice (1, 2, 3, or 4): ")

if choice == '1':
    visualize_predictions(best_model, test_ds, class_names, num_samples=5)
elif choice == '2':
    show_confusion_matrix_with_predictions(best_model, test_ds, class_names)
elif choice == '3':
    show_misclassifications(best_model, test_ds, class_names, num_samples=3)
else:
    show_accuracy_summary(best_model, test_ds, class_names)

print("\n✓ All visualizations complete!")

In [ ]:
# ============================================================
# SIMPLE PREDICTION VISUALIZATION (WORKS 100%)
# ============================================================

def visualize_predictions(model, dataset, class_names, num_samples=5):
    """
    Simple visualization of model predictions with black text
    """
    print("\n" + "="*60)
    print("MODEL PREDICTION VISUALIZATION")
    print("="*60)

    # Collect samples
    samples = []
    for images, labels in dataset.take(2):
        for i in range(min(5, len(images))):
            if len(samples) >= num_samples:
                break
            samples.append((images[i].numpy(), labels[i].numpy()))

    print(f"Displaying {len(samples)} sample predictions...\n")

    # Create figure
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    axes = axes.flatten()

    for idx, (img, true_label) in enumerate(samples):
        if idx >= 5:
            break

        # Make prediction
        img_array = np.expand_dims(img, axis=0)
        preds = model.predict(img_array, verbose=0)[0]
        pred_label = np.argmax(preds)
        confidence = preds[pred_label]

        # Normalize image for display
        img_display = (img - img.min()) / (img.max() - img.min() + 1e-10)

        # Display image
        axes[idx].imshow(img_display)

        # Set title with black text (no color coding)
        axes[idx].set_title(f'Sample {idx+1}\nTrue: {class_names[true_label]}\nPred: {class_names[pred_label]}\nConf: {confidence:.3f}',
                           color='black', fontsize=11, fontweight='bold')
        axes[idx].axis('off')

        # Add small bar plot of probabilities
        ax_inset = axes[idx].inset_axes([0.05, 0.05, 0.9, 0.2])
        y_pos = np.arange(len(class_names))
        colors = ['#FF6B6B' if i == pred_label else '#4ECDC4' for i in range(len(class_names))]
        ax_inset.barh(y_pos, preds, color=colors, height=0.5)
        ax_inset.set_yticks(y_pos)
        ax_inset.set_yticklabels([c[:8] for c in class_names], fontsize=7, color='black')
        ax_inset.set_xlim(0, 1)
        ax_inset.set_xlabel('Confidence', fontsize=7, color='black')
        ax_inset.tick_params(colors='black', labelsize=6)

        # Set axis colors to black
        ax_inset.spines['bottom'].set_color('black')
        ax_inset.spines['top'].set_color('black')
        ax_inset.spines['left'].set_color('black')
        ax_inset.spines['right'].set_color('black')

    # Hide unused subplot
    for idx in range(len(samples), 6):
        axes[idx].axis('off')

    plt.suptitle('Model Predictions on Test Samples', fontsize=14, fontweight='bold', color='black', y=1.02)
    plt.tight_layout()
    plt.show()

    # Print summary with black text (remove color codes)
    print("\n" + "="*60)
    print("PREDICTION SUMMARY")
    print("="*60)

    correct = 0
    for idx, (img, true_label) in enumerate(samples[:5]):
        img_array = np.expand_dims(img, axis=0)
        preds = model.predict(img_array, verbose=0)[0]
        pred_label = np.argmax(preds)
        confidence = preds[pred_label]

        status = "CORRECT" if true_label == pred_label else "INCORRECT"

        print(f"Sample {idx+1}: {status}")
        print(f"  True: {class_names[true_label]}")
        print(f"  Pred: {class_names[pred_label]} (conf: {confidence:.3f})")
        print(f"  Confidence distribution: {[f'{c:.2f}' for c in preds]}")
        print()

        if true_label == pred_label:
            correct += 1

    accuracy = correct / len(samples[:5])
    print(f"Accuracy on displayed samples: {correct}/{len(samples[:5])} = {accuracy:.2%}")

# ============================================================
# CONFUSION MATRIX VISUALIZATION
# ============================================================

def show_confusion_matrix_with_predictions(model, dataset, class_names):
    """
    Show confusion matrix with sample predictions (black text)
    """
    print("\n" + "="*60)
    print("CONFUSION MATRIX WITH SAMPLE PREDICTIONS")
    print("="*60)

    # Collect all predictions
    y_true = []
    y_pred = []

    for images, labels in dataset:
        preds = model.predict(images, verbose=0)
        y_true.extend(labels.numpy())
        y_pred.extend(np.argmax(preds, axis=1))

    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    # Create confusion matrix
    cm = confusion_matrix(y_true, y_pred)

    # Plot confusion matrix with black text
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names,
                annot_kws={'size': 14, 'weight': 'bold', 'color': 'black'},
                cbar_kws={'label': 'Count'})
    plt.title('Confusion Matrix', fontsize=14, fontweight='bold', color='black', pad=20)
    plt.ylabel('True Label', fontsize=12, fontweight='bold', color='black')
    plt.xlabel('Predicted Label', fontsize=12, fontweight='bold', color='black')

    # Set tick colors to black
    plt.xticks(color='black')
    plt.yticks(color='black')

    # Set colorbar label color
    cbar = plt.gcf().axes[-1]
    cbar.yaxis.label.set_color('black')
    cbar.tick_params(colors='black')

    plt.tight_layout()
    plt.show()

    # Calculate metrics
    accuracy = np.mean(y_true == y_pred)
    print(f"\nOverall Accuracy: {accuracy:.4f}")

    # Per-class accuracy
    print("\nPer-class Accuracy:")
    for i, name in enumerate(class_names):
        class_mask = y_true == i
        if np.sum(class_mask) > 0:
            class_acc = np.mean(y_pred[class_mask] == i)
            print(f"  {name}: {class_acc:.4f} ({np.sum(class_mask)} samples)")

# ============================================================
# SAMPLE MISCLASSIFICATIONS
# ============================================================

def show_misclassifications(model, dataset, class_names, num_samples=3):
    """
    Show examples where model made mistakes (black text)
    """
    print("\n" + "="*60)
    print("MISCLASSIFICATION ANALYSIS")
    print("="*60)

    misclassified = []

    for images, labels in dataset:
        preds = model.predict(images, verbose=0)
        pred_labels = np.argmax(preds, axis=1)

        for i in range(len(labels)):
            if labels[i] != pred_labels[i]:
                misclassified.append((images[i].numpy(), labels[i].numpy(), pred_labels[i], preds[i]))
                if len(misclassified) >= num_samples:
                    break
        if len(misclassified) >= num_samples:
            break

    if not misclassified:
        print("No misclassifications found! (Model is perfect on these samples)")
        return

    fig, axes = plt.subplots(1, len(misclassified), figsize=(15, 4))
    if len(misclassified) == 1:
        axes = [axes]

    for idx, (img, true_label, pred_label, probs) in enumerate(misclassified):
        img_display = (img - img.min()) / (img.max() - img.min() + 1e-10)

        axes[idx].imshow(img_display)
        axes[idx].set_title(f'True: {class_names[true_label]}\nPred: {class_names[pred_label]}\nConf: {probs[pred_label]:.3f}',
                           color='black', fontsize=10, fontweight='bold')
        axes[idx].axis('off')

        # Add border to highlight misclassification
        for spine in axes[idx].spines.values():
            spine.set_edgecolor('red')
            spine.set_linewidth(2)

    plt.suptitle('Misclassified Examples', fontsize=14, fontweight='bold', color='black', y=1.05)
    plt.tight_layout()
    plt.show()

    # Print summary
    print(f"\nFound {len(misclassified)} misclassified samples")
    for idx, (img, true_label, pred_label, probs) in enumerate(misclassified):
        print(f"  {idx+1}. True: {class_names[true_label]} → Pred: {class_names[pred_label]} (conf: {probs[pred_label]:.3f})")

# ============================================================
# ACCURACY VISUALIZATION
# ============================================================

def show_accuracy_summary(model, dataset, class_names):
    """
    Show accuracy summary with black text
    """
    print("\n" + "="*60)
    print("ACCURACY SUMMARY")
    print("="*60)

    # Collect predictions
    y_true = []
    y_pred = []

    for images, labels in dataset:
        preds = model.predict(images, verbose=0)
        y_true.extend(labels.numpy())
        y_pred.extend(np.argmax(preds, axis=1))

    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    # Calculate overall accuracy
    accuracy = np.mean(y_true == y_pred)

    # Create accuracy bar chart
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

    # Overall accuracy
    ax1.bar(['Overall Accuracy'], [accuracy], color='#4ECDC4', edgecolor='black', linewidth=2)
    ax1.set_ylim(0, 1)
    ax1.set_ylabel('Accuracy', fontsize=12, fontweight='bold', color='black')
    ax1.set_title('Overall Model Accuracy', fontsize=13, fontweight='bold', color='black', pad=20)

    # Add value label
    ax1.text(0, accuracy + 0.02, f'{accuracy:.3f}', ha='center', fontsize=12, fontweight='bold', color='black')

    # Set tick colors
    ax1.tick_params(colors='black')
    for spine in ax1.spines.values():
        spine.set_color('black')

    # Per-class accuracy
    class_accuracies = []
    for i in range(len(class_names)):
        class_mask = y_true == i
        if np.sum(class_mask) > 0:
            class_acc = np.mean(y_pred[class_mask] == i)
            class_accuracies.append(class_acc)
        else:
            class_accuracies.append(0)

    bars = ax2.bar(class_names, class_accuracies, color=['#FF6B6B', '#4ECDC4', '#FFD93D', '#6C5CE7'],
                   edgecolor='black', linewidth=2)
    ax2.set_ylim(0, 1)
    ax2.set_ylabel('Accuracy', fontsize=12, fontweight='bold', color='black')
    ax2.set_title('Per-Class Accuracy', fontsize=13, fontweight='bold', color='black', pad=20)

    # Add value labels
    for bar, acc in zip(bars, class_accuracies):
        ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                f'{acc:.3f}', ha='center', fontsize=10, fontweight='bold', color='black')

    # Set tick colors
    ax2.tick_params(colors='black')
    for spine in ax2.spines.values():
        spine.set_color('black')
    ax2.set_xticklabels(class_names, rotation=45, ha='right')

    plt.suptitle('Model Performance Analysis', fontsize=14, fontweight='bold', color='black', y=1.05)
    plt.tight_layout()
    plt.show()

    print(f"\nOverall Test Accuracy: {accuracy:.4f}")
    print("\nPer-class Accuracies:")
    for i, name in enumerate(class_names):
        print(f"  {name}: {class_accuracies[i]:.4f}")

# ============================================================
# RUN VISUALIZATIONS
# ============================================================

print("\nChoose visualization:")
print("1. Prediction Samples (with confidence bars)")
print("2. Confusion Matrix")
print("3. Misclassification Analysis")
print("4. Accuracy Summary")

choice = input("\nEnter your choice (1, 2, 3, or 4): ")

if choice == '1':
    visualize_predictions(best_model, test_ds, class_names, num_samples=5)
elif choice == '2':
    show_confusion_matrix_with_predictions(best_model, test_ds, class_names)
elif choice == '3':
    show_misclassifications(best_model, test_ds, class_names, num_samples=3)
else:
    show_accuracy_summary(best_model, test_ds, class_names)

print("\n✓ All visualizations complete!")

In [ ]:
# ============================================================
# LIME VISUALIZATION
# ============================================================

try:
    from lime import lime_image
    from skimage.segmentation import mark_boundaries
    LIME_AVAILABLE = True
except ImportError:
    print("Installing LIME...")
    !pip install lime -q
    from lime import lime_image
    from skimage.segmentation import mark_boundaries
    LIME_AVAILABLE = True

def setup_lime_explainer():
    """
    Setup LIME image explainer
    """
    return lime_image.LimeImageExplainer()

def predict_fn(images, model):
    """
    Prediction function for LIME
    """
    # Ensure images are in correct format
    if len(images.shape) == 3:
        images = np.expand_dims(images, axis=0)

    # Preprocess images (model expects preprocessed input)
    processed_images = []
    for img in images:
        # Normalize to [0,1] range if needed
        if img.max() > 1.0:
            img = img / 255.0
        # Apply model-specific preprocessing
        processed = preprocess_input(img * 255)  # Convert back to [0,255] range for preprocess_input
        processed_images.append(processed)

    processed_images = np.array(processed_images)

    # Get predictions
    predictions = model.predict(processed_images, verbose=0)

    return predictions

def generate_lime_explanation(explainer, image, model, class_names, top_labels=3, num_samples=1000):
    """
    Generate LIME explanation for an image
    """
    # Ensure image is in correct format (0-1 range)
    if image.max() > 1.0:
        image = image / 255.0

    # Generate explanation
    explanation = explainer.explain_instance(
        image,
        lambda x: predict_fn(x, model),
        top_labels=top_labels,
        hide_color=0,
        num_samples=num_samples,
        random_seed=42
    )

    return explanation

def visualize_lime_explanation(image, explanation, true_label, pred_label, class_names, sample_idx):
    """
    Visualize LIME explanation with black text
    """
    # Get the explanation for the predicted class
    pred_class_idx = pred_label

    # Get image and mask for positive regions only
    temp_pos, mask_pos = explanation.get_image_and_mask(
        pred_class_idx,
        positive_only=True,
        num_features=5,
        hide_rest=False
    )

    # Get image and mask for positive and negative regions
    temp_pos_neg, mask_pos_neg = explanation.get_image_and_mask(
        pred_class_idx,
        positive_only=False,
        num_features=5,
        hide_rest=False
    )

    # Create figure
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))

    # Original image
    img_display = (image - image.min()) / (image.max() - image.min() + 1e-10)
    axes[0, 0].imshow(img_display)
    axes[0, 0].set_title(f'Original Image\nTrue: {class_names[true_label]}',
                         fontsize=12, fontweight='bold', color='black')
    axes[0, 0].axis('off')

    # LIME with positive features only
    axes[0, 1].imshow(mark_boundaries(temp_pos, mask_pos))
    axes[0, 1].set_title(f'LIME - Positive Features\n(Important regions for prediction)',
                         fontsize=12, fontweight='bold', color='black')
    axes[0, 1].axis('off')

    # LIME with positive and negative features
    axes[0, 2].imshow(mark_boundaries(temp_pos_neg, mask_pos_neg))
    axes[0, 2].set_title(f'LIME - Positive (Green) & Negative (Red)\nPred: {class_names[pred_label]}',
                         fontsize=12, fontweight='bold', color='black')
    axes[0, 2].axis('off')

    # Get feature weights
    features = explanation.local_exp[pred_class_idx]
    feature_names = [f'Feature {f[0]}' for f in features]
    feature_weights = [f[1] for f in features]

    # Sort by absolute weight
    sorted_idx = np.argsort(np.abs(feature_weights))[::-1]
    top_features = sorted_idx[:10]

    # Plot feature importance
    y_pos = np.arange(len(top_features))
    axes[1, 0].barh(y_pos, [feature_weights[i] for i in top_features],
                    color=['green' if feature_weights[i] > 0 else 'red' for i in top_features])
    axes[1, 0].set_yticks(y_pos)
    axes[1, 0].set_yticklabels([f'F{i}' for i in top_features], fontsize=9)
    axes[1, 0].set_xlabel('Weight', fontsize=11, color='black')
    axes[1, 0].set_title('Top Feature Weights\n(Green: Positive, Red: Negative)',
                         fontsize=12, fontweight='bold', color='black')
    axes[1, 0].axvline(x=0, color='black', linestyle='-', linewidth=0.5)

    # Plot segmentation overlay
    axes[1, 1].imshow(img_display)

    # Get segmentation
    segmentation = explanation.segments

    # Overlay segmentation boundaries
    from skimage.segmentation import find_boundaries
    boundaries = find_boundaries(segmentation, mode='outer')
    axes[1, 1].imshow(np.ma.masked_where(~boundaries, boundaries),
                      cmap='gray', alpha=0.5)
    axes[1, 1].set_title('Image Segmentation\n(Superpixels)',
                         fontsize=12, fontweight='bold', color='black')
    axes[1, 1].axis('off')

    # Prediction probabilities
    preds = predict_fn(np.expand_dims(image, axis=0), model)[0]
    y_pos_prob = np.arange(len(class_names))
    colors = ['#4ECDC4' if i != pred_label else '#FF6B6B' for i in range(len(class_names))]
    axes[1, 2].barh(y_pos_prob, preds, color=colors)
    axes[1, 2].set_yticks(y_pos_prob)
    axes[1, 2].set_yticklabels(class_names, fontsize=9, color='black')
    axes[1, 2].set_xlim(0, 1)
    axes[1, 2].set_xlabel('Probability', fontsize=11, color='black')
    axes[1, 2].set_title('Prediction Probabilities',
                         fontsize=12, fontweight='bold', color='black')

    # Add confidence value
    axes[1, 2].text(0.5, -0.15, f'Confidence: {preds[pred_label]:.3f}',
                   transform=axes[1, 2].transAxes, ha='center',
                   fontsize=11, fontweight='bold', color='black')

    plt.suptitle(f'LIME Explanation - Sample {sample_idx}',
                 fontsize=14, fontweight='bold', color='black', y=1.02)
    plt.tight_layout()
    plt.show()

    # Print analysis
    print(f"\nLIME Analysis - Sample {sample_idx}:")
    print(f"  True class: {class_names[true_label]}")
    print(f"  Predicted class: {class_names[pred_label]}")
    print(f"  Confidence: {preds[pred_label]:.3f}")
    print(f"  Number of superpixels: {len(np.unique(explanation.segments))}")
    print("\n  Top influencing features:")
    for i, (feat_idx, weight) in enumerate(zip(top_features[:5], [feature_weights[i] for i in top_features[:5]])):
        direction = "SUPPORTS" if weight > 0 else "AGAINST"
        print(f"    {i+1}. Feature {feat_idx}: {weight:+.4f} ({direction})")
    print("-" * 60)

def run_lime_visualization(model, dataset, class_names, num_samples=3):
    """
    Run LIME visualization on sample images
    """
    print("\n" + "="*60)
    print("LIME EXPLANATION VISUALIZATION")
    print("="*60)

    # Setup LIME explainer
    explainer = setup_lime_explainer()
    print("✓ LIME explainer initialized")

    # Collect samples
    samples = []
    for images, labels in dataset.take(2):
        for i in range(min(3, len(images))):
            if len(samples) >= num_samples:
                break
            samples.append((images[i].numpy(), labels[i].numpy()))

    print(f"Processing {len(samples)} samples...\n")

    for idx, (img, true_label) in enumerate(samples):
        print(f"Generating LIME explanation for Sample {idx+1}...")

        # Prepare image (convert to 0-1 range for LIME)
        img_normalized = (img - img.min()) / (img.max() - img.min() + 1e-10)

        # Get prediction
        img_array = np.expand_dims(img, axis=0)
        preds = model.predict(img_array, verbose=0)[0]
        pred_label = np.argmax(preds)

        # Generate LIME explanation
        try:
            explanation = generate_lime_explanation(
                explainer,
                img_normalized,
                model,
                class_names,
                top_labels=3,
                num_samples=500  # Reduced for faster execution
            )

            # Visualize
            visualize_lime_explanation(
                img_normalized,
                explanation,
                true_label,
                pred_label,
                class_names,
                idx+1
            )

        except Exception as e:
            print(f"  Error generating LIME explanation: {e}")
            continue

    print("\n✓ LIME visualization complete!")

def run_lime_comparison(model, dataset, class_names, num_samples=2):
    """
    Compare LIME explanations for correct vs incorrect predictions
    """
    print("\n" + "="*60)
    print("LIME COMPARISON: CORRECT VS INCORRECT PREDICTIONS")
    print("="*60)

    explainer = setup_lime_explainer()

    # Collect correct and incorrect predictions
    correct_samples = []
    incorrect_samples = []

    for images, labels in dataset:
        preds = model.predict(images, verbose=0)
        pred_labels = np.argmax(preds, axis=1)

        for i in range(len(labels)):
            if len(correct_samples) >= num_samples and len(incorrect_samples) >= num_samples:
                break

            if labels[i] == pred_labels[i] and len(correct_samples) < num_samples:
                correct_samples.append((images[i].numpy(), labels[i].numpy(), pred_labels[i], preds[i]))
            elif labels[i] != pred_labels[i] and len(incorrect_samples) < num_samples:
                incorrect_samples.append((images[i].numpy(), labels[i].numpy(), pred_labels[i], preds[i]))

        if len(correct_samples) >= num_samples and len(incorrect_samples) >= num_samples:
            break

    # Show correct predictions
    if correct_samples:
        print("\n" + "-"*40)
        print("CORRECT PREDICTIONS")
        print("-"*40)

        for idx, (img, true_label, pred_label, probs) in enumerate(correct_samples):
            print(f"\nCorrect Prediction {idx+1}:")
            print(f"  True: {class_names[true_label]} → Pred: {class_names[pred_label]} (conf: {probs[pred_label]:.3f})")

            img_normalized = (img - img.min()) / (img.max() - img.min() + 1e-10)

            try:
                explanation = generate_lime_explanation(
                    explainer, img_normalized, model, class_names,
                    top_labels=3, num_samples=300
                )

                temp, mask = explanation.get_image_and_mask(
                    pred_label, positive_only=True, num_features=5, hide_rest=False
                )

                fig, axes = plt.subplots(1, 2, figsize=(10, 4))

                axes[0].imshow(img_normalized)
                axes[0].set_title(f'Original\nTrue: {class_names[true_label]}',
                                 fontsize=11, fontweight='bold', color='black')
                axes[0].axis('off')

                axes[1].imshow(mark_boundaries(temp, mask))
                axes[1].set_title(f'LIME Explanation\n(Important regions)',
                                 fontsize=11, fontweight='bold', color='black')
                axes[1].axis('off')

                plt.suptitle(f'Correct Prediction - {class_names[true_label]}',
                           fontsize=12, fontweight='bold', color='black')
                plt.tight_layout()
                plt.show()

            except Exception as e:
                print(f"  Error: {e}")

    # Show incorrect predictions
    if incorrect_samples:
        print("\n" + "-"*40)
        print("INCORRECT PREDICTIONS")
        print("-"*40)

        for idx, (img, true_label, pred_label, probs) in enumerate(incorrect_samples):
            print(f"\nIncorrect Prediction {idx+1}:")
            print(f"  True: {class_names[true_label]} → Pred: {class_names[pred_label]} (conf: {probs[pred_label]:.3f})")

            img_normalized = (img - img.min()) / (img.max() - img.min() + 1e-10)

            try:
                explanation = generate_lime_explanation(
                    explainer, img_normalized, model, class_names,
                    top_labels=3, num_samples=300
                )

                temp, mask = explanation.get_image_and_mask(
                    pred_label, positive_only=True, num_features=5, hide_rest=False
                )

                fig, axes = plt.subplots(1, 2, figsize=(10, 4))

                axes[0].imshow(img_normalized)
                axes[0].set_title(f'Original\nTrue: {class_names[true_label]}',
                                 fontsize=11, fontweight='bold', color='black')
                axes[0].axis('off')

                axes[1].imshow(mark_boundaries(temp, mask))
                axes[1].set_title(f'LIME Explanation\n(Why it thought {class_names[pred_label]})',
                                 fontsize=11, fontweight='bold', color='black')
                axes[1].axis('off')

                # Add red border for incorrect prediction
                for spine in axes[1].spines.values():
                    spine.set_edgecolor('red')
                    spine.set_linewidth(2)

                plt.suptitle(f'Incorrect Prediction - {class_names[true_label]} → {class_names[pred_label]}',
                           fontsize=12, fontweight='bold', color='black')
                plt.tight_layout()
                plt.show()

            except Exception as e:
                print(f"  Error: {e}")

def run_simple_lime(model, dataset, class_names, num_samples=2):
    """
    Simplified LIME visualization (faster and more reliable)
    """
    print("\n" + "="*60)
    print("SIMPLE LIME VISUALIZATION")
    print("="*60)

    explainer = setup_lime_explainer()

    samples = []
    for images, labels in dataset.take(2):
        for i in range(min(2, len(images))):
            if len(samples) >= num_samples:
                break
            samples.append((images[i].numpy(), labels[i].numpy()))

    for idx, (img, true_label) in enumerate(samples):
        print(f"\nProcessing Sample {idx+1}...")

        img_normalized = (img - img.min()) / (img.max() - img.min() + 1e-10)

        img_array = np.expand_dims(img, axis=0)
        preds = model.predict(img_array, verbose=0)[0]
        pred_label = np.argmax(preds)

        try:
            # Generate explanation with fewer samples for speed
            explanation = explainer.explain_instance(
                img_normalized,
                lambda x: predict_fn(x, model),
                top_labels=1,
                hide_color=0,
                num_samples=200,
                random_seed=42
            )

            # Get visualization
            temp, mask = explanation.get_image_and_mask(
                pred_label,
                positive_only=True,
                num_features=5,
                hide_rest=False
            )

            # Plot
            fig, axes = plt.subplots(1, 3, figsize=(15, 4))

            axes[0].imshow(img_normalized)
            axes[0].set_title(f'Original\nTrue: {class_names[true_label]}',
                             fontsize=11, fontweight='bold', color='black')
            axes[0].axis('off')

            axes[1].imshow(mark_boundaries(temp, mask))
            axes[1].set_title(f'LIME - Important Regions\nPred: {class_names[pred_label]}',
                             fontsize=11, fontweight='bold', color='black')
            axes[1].axis('off')

            # Confidence bar
            colors = ['#4ECDC4' if i != pred_label else '#FF6B6B' for i in range(len(class_names))]
            axes[2].barh(class_names, preds, color=colors)
            axes[2].set_xlim(0, 1)
            axes[2].set_xlabel('Confidence', fontsize=11, color='black')
            axes[2].set_title(f'Confidence: {preds[pred_label]:.3f}',
                             fontsize=11, fontweight='bold', color='black')

            plt.suptitle(f'LIME Explanation - Sample {idx+1}',
                        fontsize=13, fontweight='bold', color='black')
            plt.tight_layout()
            plt.show()

        except Exception as e:
            print(f"  Error: {e}")

# ============================================================
# ADD TO YOUR EXISTING MENU
# ============================================================

# Add this to your existing menu options
print("\nChoose visualization:")
print("1. Prediction Samples (with confidence bars)")
print("2. Confusion Matrix")
print("3. Misclassification Analysis")
print("4. Accuracy Summary")
print("5. LIME Visualization")
print("6. LIME Comparison (Correct vs Incorrect)")
print("7. Simple LIME (Fast)")

choice = input("\nEnter your choice (1-7): ")

if choice == '1':
    visualize_predictions(best_model, test_ds, class_names, num_samples=5)
elif choice == '2':
    show_confusion_matrix_with_predictions(best_model, test_ds, class_names)
elif choice == '3':
    show_misclassifications(best_model, test_ds, class_names, num_samples=3)
elif choice == '4':
    show_accuracy_summary(best_model, test_ds, class_names)
elif choice == '5':
    run_lime_visualization(best_model, test_ds, class_names, num_samples=2)
elif choice == '6':
    run_lime_comparison(best_model, test_ds, class_names, num_samples=1)
else:
    run_simple_lime(best_model, test_ds, class_names, num_samples=2)

print("\n✓ All visualizations complete!")

In [ ]:
# ============================================================
# LIME VISUALIZATION
# ============================================================

try:
    from lime import lime_image
    from skimage.segmentation import mark_boundaries
    LIME_AVAILABLE = True
except ImportError:
    print("Installing LIME...")
    !pip install lime -q
    from lime import lime_image
    from skimage.segmentation import mark_boundaries
    LIME_AVAILABLE = True

def setup_lime_explainer():
    """
    Setup LIME image explainer
    """
    return lime_image.LimeImageExplainer()

def predict_fn(images, model):
    """
    Prediction function for LIME
    """
    # Ensure images are in correct format
    if len(images.shape) == 3:
        images = np.expand_dims(images, axis=0)

    # Preprocess images (model expects preprocessed input)
    processed_images = []
    for img in images:
        # Normalize to [0,1] range if needed
        if img.max() > 1.0:
            img = img / 255.0
        # Apply model-specific preprocessing
        processed = preprocess_input(img * 255)  # Convert back to [0,255] range for preprocess_input
        processed_images.append(processed)

    processed_images = np.array(processed_images)

    # Get predictions
    predictions = model.predict(processed_images, verbose=0)

    return predictions

def generate_lime_explanation(explainer, image, model, class_names, top_labels=3, num_samples=1000):
    """
    Generate LIME explanation for an image
    """
    # Ensure image is in correct format (0-1 range)
    if image.max() > 1.0:
        image = image / 255.0

    # Generate explanation
    explanation = explainer.explain_instance(
        image,
        lambda x: predict_fn(x, model),
        top_labels=top_labels,
        hide_color=0,
        num_samples=num_samples,
        random_seed=42
    )

    return explanation

def visualize_lime_explanation(image, explanation, true_label, pred_label, class_names, sample_idx):
    """
    Visualize LIME explanation with black text
    """
    # Get the explanation for the predicted class
    pred_class_idx = pred_label

    # Get image and mask for positive regions only
    temp_pos, mask_pos = explanation.get_image_and_mask(
        pred_class_idx,
        positive_only=True,
        num_features=5,
        hide_rest=False
    )

    # Get image and mask for positive and negative regions
    temp_pos_neg, mask_pos_neg = explanation.get_image_and_mask(
        pred_class_idx,
        positive_only=False,
        num_features=5,
        hide_rest=False
    )

    # Create figure
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))

    # Original image
    img_display = (image - image.min()) / (image.max() - image.min() + 1e-10)
    axes[0, 0].imshow(img_display)
    axes[0, 0].set_title(f'Original Image\nTrue: {class_names[true_label]}',
                         fontsize=12, fontweight='bold', color='black')
    axes[0, 0].axis('off')

    # LIME with positive features only
    axes[0, 1].imshow(mark_boundaries(temp_pos, mask_pos))
    axes[0, 1].set_title(f'LIME - Positive Features\n(Important regions for prediction)',
                         fontsize=12, fontweight='bold', color='black')
    axes[0, 1].axis('off')

    # LIME with positive and negative features
    axes[0, 2].imshow(mark_boundaries(temp_pos_neg, mask_pos_neg))
    axes[0, 2].set_title(f'LIME - Positive (Green) & Negative (Red)\nPred: {class_names[pred_label]}',
                         fontsize=12, fontweight='bold', color='black')
    axes[0, 2].axis('off')

    # Get feature weights
    features = explanation.local_exp[pred_class_idx]
    feature_names = [f'Feature {f[0]}' for f in features]
    feature_weights = [f[1] for f in features]

    # Sort by absolute weight
    sorted_idx = np.argsort(np.abs(feature_weights))[::-1]
    top_features = sorted_idx[:10]

    # Plot feature importance
    y_pos = np.arange(len(top_features))
    axes[1, 0].barh(y_pos, [feature_weights[i] for i in top_features],
                    color=['green' if feature_weights[i] > 0 else 'red' for i in top_features])
    axes[1, 0].set_yticks(y_pos)
    axes[1, 0].set_yticklabels([f'F{i}' for i in top_features], fontsize=9)
    axes[1, 0].set_xlabel('Weight', fontsize=11, color='black')
    axes[1, 0].set_title('Top Feature Weights\n(Green: Positive, Red: Negative)',
                         fontsize=12, fontweight='bold', color='black')
    axes[1, 0].axvline(x=0, color='black', linestyle='-', linewidth=0.5)

    # Plot segmentation overlay
    axes[1, 1].imshow(img_display)

    # Get segmentation
    segmentation = explanation.segments

    # Overlay segmentation boundaries
    from skimage.segmentation import find_boundaries
    boundaries = find_boundaries(segmentation, mode='outer')
    axes[1, 1].imshow(np.ma.masked_where(~boundaries, boundaries),
                      cmap='gray', alpha=0.5)
    axes[1, 1].set_title('Image Segmentation\n(Superpixels)',
                         fontsize=12, fontweight='bold', color='black')
    axes[1, 1].axis('off')

    # Prediction probabilities
    preds = predict_fn(np.expand_dims(image, axis=0), model)[0]
    y_pos_prob = np.arange(len(class_names))
    colors = ['#4ECDC4' if i != pred_label else '#FF6B6B' for i in range(len(class_names))]
    axes[1, 2].barh(y_pos_prob, preds, color=colors)
    axes[1, 2].set_yticks(y_pos_prob)
    axes[1, 2].set_yticklabels(class_names, fontsize=9, color='black')
    axes[1, 2].set_xlim(0, 1)
    axes[1, 2].set_xlabel('Probability', fontsize=11, color='black')
    axes[1, 2].set_title('Prediction Probabilities',
                         fontsize=12, fontweight='bold', color='black')

    # Add confidence value
    axes[1, 2].text(0.5, -0.15, f'Confidence: {preds[pred_label]:.3f}',
                   transform=axes[1, 2].transAxes, ha='center',
                   fontsize=11, fontweight='bold', color='black')

    plt.suptitle(f'LIME Explanation - Sample {sample_idx}',
                 fontsize=14, fontweight='bold', color='black', y=1.02)
    plt.tight_layout()
    plt.show()

    # Print analysis
    print(f"\nLIME Analysis - Sample {sample_idx}:")
    print(f"  True class: {class_names[true_label]}")
    print(f"  Predicted class: {class_names[pred_label]}")
    print(f"  Confidence: {preds[pred_label]:.3f}")
    print(f"  Number of superpixels: {len(np.unique(explanation.segments))}")
    print("\n  Top influencing features:")
    for i, (feat_idx, weight) in enumerate(zip(top_features[:5], [feature_weights[i] for i in top_features[:5]])):
        direction = "SUPPORTS" if weight > 0 else "AGAINST"
        print(f"    {i+1}. Feature {feat_idx}: {weight:+.4f} ({direction})")
    print("-" * 60)

def run_lime_visualization(model, dataset, class_names, num_samples=3):
    """
    Run LIME visualization on sample images
    """
    print("\n" + "="*60)
    print("LIME EXPLANATION VISUALIZATION")
    print("="*60)

    # Setup LIME explainer
    explainer = setup_lime_explainer()
    print("✓ LIME explainer initialized")

    # Collect samples
    samples = []
    for images, labels in dataset.take(2):
        for i in range(min(3, len(images))):
            if len(samples) >= num_samples:
                break
            samples.append((images[i].numpy(), labels[i].numpy()))

    print(f"Processing {len(samples)} samples...\n")

    for idx, (img, true_label) in enumerate(samples):
        print(f"Generating LIME explanation for Sample {idx+1}...")

        # Prepare image (convert to 0-1 range for LIME)
        img_normalized = (img - img.min()) / (img.max() - img.min() + 1e-10)

        # Get prediction
        img_array = np.expand_dims(img, axis=0)
        preds = model.predict(img_array, verbose=0)[0]
        pred_label = np.argmax(preds)

        # Generate LIME explanation
        try:
            explanation = generate_lime_explanation(
                explainer,
                img_normalized,
                model,
                class_names,
                top_labels=3,
                num_samples=500  # Reduced for faster execution
            )

            # Visualize
            visualize_lime_explanation(
                img_normalized,
                explanation,
                true_label,
                pred_label,
                class_names,
                idx+1
            )

        except Exception as e:
            print(f"  Error generating LIME explanation: {e}")
            continue

    print("\n✓ LIME visualization complete!")

def run_lime_comparison(model, dataset, class_names, num_samples=2):
    """
    Compare LIME explanations for correct vs incorrect predictions
    """
    print("\n" + "="*60)
    print("LIME COMPARISON: CORRECT VS INCORRECT PREDICTIONS")
    print("="*60)

    explainer = setup_lime_explainer()

    # Collect correct and incorrect predictions
    correct_samples = []
    incorrect_samples = []

    for images, labels in dataset:
        preds = model.predict(images, verbose=0)
        pred_labels = np.argmax(preds, axis=1)

        for i in range(len(labels)):
            if len(correct_samples) >= num_samples and len(incorrect_samples) >= num_samples:
                break

            if labels[i] == pred_labels[i] and len(correct_samples) < num_samples:
                correct_samples.append((images[i].numpy(), labels[i].numpy(), pred_labels[i], preds[i]))
            elif labels[i] != pred_labels[i] and len(incorrect_samples) < num_samples:
                incorrect_samples.append((images[i].numpy(), labels[i].numpy(), pred_labels[i], preds[i]))

        if len(correct_samples) >= num_samples and len(incorrect_samples) >= num_samples:
            break

    # Show correct predictions
    if correct_samples:
        print("\n" + "-"*40)
        print("CORRECT PREDICTIONS")
        print("-"*40)

        for idx, (img, true_label, pred_label, probs) in enumerate(correct_samples):
            print(f"\nCorrect Prediction {idx+1}:")
            print(f"  True: {class_names[true_label]} → Pred: {class_names[pred_label]} (conf: {probs[pred_label]:.3f})")

            img_normalized = (img - img.min()) / (img.max() - img.min() + 1e-10)

            try:
                explanation = generate_lime_explanation(
                    explainer, img_normalized, model, class_names,
                    top_labels=3, num_samples=300
                )

                temp, mask = explanation.get_image_and_mask(
                    pred_label, positive_only=True, num_features=5, hide_rest=False
                )

                fig, axes = plt.subplots(1, 2, figsize=(10, 4))

                axes[0].imshow(img_normalized)
                axes[0].set_title(f'Original\nTrue: {class_names[true_label]}',
                                 fontsize=11, fontweight='bold', color='black')
                axes[0].axis('off')

                axes[1].imshow(mark_boundaries(temp, mask))
                axes[1].set_title(f'LIME Explanation\n(Important regions)',
                                 fontsize=11, fontweight='bold', color='black')
                axes[1].axis('off')

                plt.suptitle(f'Correct Prediction - {class_names[true_label]}',
                           fontsize=12, fontweight='bold', color='black')
                plt.tight_layout()
                plt.show()

            except Exception as e:
                print(f"  Error: {e}")

    # Show incorrect predictions
    if incorrect_samples:
        print("\n" + "-"*40)
        print("INCORRECT PREDICTIONS")
        print("-"*40)

        for idx, (img, true_label, pred_label, probs) in enumerate(incorrect_samples):
            print(f"\nIncorrect Prediction {idx+1}:")
            print(f"  True: {class_names[true_label]} → Pred: {class_names[pred_label]} (conf: {probs[pred_label]:.3f})")

            img_normalized = (img - img.min()) / (img.max() - img.min() + 1e-10)

            try:
                explanation = generate_lime_explanation(
                    explainer, img_normalized, model, class_names,
                    top_labels=3, num_samples=300
                )

                temp, mask = explanation.get_image_and_mask(
                    pred_label, positive_only=True, num_features=5, hide_rest=False
                )

                fig, axes = plt.subplots(1, 2, figsize=(10, 4))

                axes[0].imshow(img_normalized)
                axes[0].set_title(f'Original\nTrue: {class_names[true_label]}',
                                 fontsize=11, fontweight='bold', color='black')
                axes[0].axis('off')

                axes[1].imshow(mark_boundaries(temp, mask))
                axes[1].set_title(f'LIME Explanation\n(Why it thought {class_names[pred_label]})',
                                 fontsize=11, fontweight='bold', color='black')
                axes[1].axis('off')

                # Add red border for incorrect prediction
                for spine in axes[1].spines.values():
                    spine.set_edgecolor('red')
                    spine.set_linewidth(2)

                plt.suptitle(f'Incorrect Prediction - {class_names[true_label]} → {class_names[pred_label]}',
                           fontsize=12, fontweight='bold', color='black')
                plt.tight_layout()
                plt.show()

            except Exception as e:
                print(f"  Error: {e}")

def run_simple_lime(model, dataset, class_names, num_samples=2):
    """
    Simplified LIME visualization (faster and more reliable)
    """
    print("\n" + "="*60)
    print("SIMPLE LIME VISUALIZATION")
    print("="*60)

    explainer = setup_lime_explainer()

    samples = []
    for images, labels in dataset.take(2):
        for i in range(min(2, len(images))):
            if len(samples) >= num_samples:
                break
            samples.append((images[i].numpy(), labels[i].numpy()))

    for idx, (img, true_label) in enumerate(samples):
        print(f"\nProcessing Sample {idx+1}...")

        img_normalized = (img - img.min()) / (img.max() - img.min() + 1e-10)

        img_array = np.expand_dims(img, axis=0)
        preds = model.predict(img_array, verbose=0)[0]
        pred_label = np.argmax(preds)

        try:
            # Generate explanation with fewer samples for speed
            explanation = explainer.explain_instance(
                img_normalized,
                lambda x: predict_fn(x, model),
                top_labels=1,
                hide_color=0,
                num_samples=200,
                random_seed=42
            )

            # Get visualization
            temp, mask = explanation.get_image_and_mask(
                pred_label,
                positive_only=True,
                num_features=5,
                hide_rest=False
            )

            # Plot
            fig, axes = plt.subplots(1, 3, figsize=(15, 4))

            axes[0].imshow(img_normalized)
            axes[0].set_title(f'Original\nTrue: {class_names[true_label]}',
                             fontsize=11, fontweight='bold', color='black')
            axes[0].axis('off')

            axes[1].imshow(mark_boundaries(temp, mask))
            axes[1].set_title(f'LIME - Important Regions\nPred: {class_names[pred_label]}',
                             fontsize=11, fontweight='bold', color='black')
            axes[1].axis('off')

            # Confidence bar
            colors = ['#4ECDC4' if i != pred_label else '#FF6B6B' for i in range(len(class_names))]
            axes[2].barh(class_names, preds, color=colors)
            axes[2].set_xlim(0, 1)
            axes[2].set_xlabel('Confidence', fontsize=11, color='black')
            axes[2].set_title(f'Confidence: {preds[pred_label]:.3f}',
                             fontsize=11, fontweight='bold', color='black')

            plt.suptitle(f'LIME Explanation - Sample {idx+1}',
                        fontsize=13, fontweight='bold', color='black')
            plt.tight_layout()
            plt.show()

        except Exception as e:
            print(f"  Error: {e}")

# ============================================================
# ADD TO YOUR EXISTING MENU
# ============================================================

# Add this to your existing menu options
print("\nChoose visualization:")
print("1. Prediction Samples (with confidence bars)")
print("2. Confusion Matrix")
print("3. Misclassification Analysis")
print("4. Accuracy Summary")
print("5. LIME Visualization")
print("6. LIME Comparison (Correct vs Incorrect)")
print("7. Simple LIME (Fast)")

choice = input("\nEnter your choice (1-7): ")

if choice == '1':
    visualize_predictions(best_model, test_ds, class_names, num_samples=5)
elif choice == '2':
    show_confusion_matrix_with_predictions(best_model, test_ds, class_names)
elif choice == '3':
    show_misclassifications(best_model, test_ds, class_names, num_samples=3)
elif choice == '4':
    show_accuracy_summary(best_model, test_ds, class_names)
elif choice == '5':
    run_lime_visualization(best_model, test_ds, class_names, num_samples=2)
elif choice == '6':
    run_lime_comparison(best_model, test_ds, class_names, num_samples=1)
else:
    run_simple_lime(best_model, test_ds, class_names, num_samples=2)

print("\n✓ All visualizations complete!")

In [ ]:
# ============================================================
# LIME VISUALIZATION
# ============================================================

try:
    from lime import lime_image
    from skimage.segmentation import mark_boundaries
    LIME_AVAILABLE = True
except ImportError:
    print("Installing LIME...")
    !pip install lime -q
    from lime import lime_image
    from skimage.segmentation import mark_boundaries
    LIME_AVAILABLE = True

def setup_lime_explainer():
    """
    Setup LIME image explainer
    """
    return lime_image.LimeImageExplainer()

def predict_fn(images, model):
    """
    Prediction function for LIME
    """
    # Ensure images are in correct format
    if len(images.shape) == 3:
        images = np.expand_dims(images, axis=0)

    # Preprocess images (model expects preprocessed input)
    processed_images = []
    for img in images:
        # Normalize to [0,1] range if needed
        if img.max() > 1.0:
            img = img / 255.0
        # Apply model-specific preprocessing
        processed = preprocess_input(img * 255)  # Convert back to [0,255] range for preprocess_input
        processed_images.append(processed)

    processed_images = np.array(processed_images)

    # Get predictions
    predictions = model.predict(processed_images, verbose=0)

    return predictions

def generate_lime_explanation(explainer, image, model, class_names, top_labels=3, num_samples=1000):
    """
    Generate LIME explanation for an image
    """
    # Ensure image is in correct format (0-1 range)
    if image.max() > 1.0:
        image = image / 255.0

    # Generate explanation
    explanation = explainer.explain_instance(
        image,
        lambda x: predict_fn(x, model),
        top_labels=top_labels,
        hide_color=0,
        num_samples=num_samples,
        random_seed=42
    )

    return explanation

def visualize_lime_explanation(image, explanation, true_label, pred_label, class_names, sample_idx):
    """
    Visualize LIME explanation with black text
    """
    # Get the explanation for the predicted class
    pred_class_idx = pred_label

    # Get image and mask for positive regions only
    temp_pos, mask_pos = explanation.get_image_and_mask(
        pred_class_idx,
        positive_only=True,
        num_features=5,
        hide_rest=False
    )

    # Get image and mask for positive and negative regions
    temp_pos_neg, mask_pos_neg = explanation.get_image_and_mask(
        pred_class_idx,
        positive_only=False,
        num_features=5,
        hide_rest=False
    )

    # Create figure
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))

    # Original image
    img_display = (image - image.min()) / (image.max() - image.min() + 1e-10)
    axes[0, 0].imshow(img_display)
    axes[0, 0].set_title(f'Original Image\nTrue: {class_names[true_label]}',
                         fontsize=12, fontweight='bold', color='black')
    axes[0, 0].axis('off')

    # LIME with positive features only
    axes[0, 1].imshow(mark_boundaries(temp_pos, mask_pos))
    axes[0, 1].set_title(f'LIME - Positive Features\n(Important regions for prediction)',
                         fontsize=12, fontweight='bold', color='black')
    axes[0, 1].axis('off')

    # LIME with positive and negative features
    axes[0, 2].imshow(mark_boundaries(temp_pos_neg, mask_pos_neg))
    axes[0, 2].set_title(f'LIME - Positive (Green) & Negative (Red)\nPred: {class_names[pred_label]}',
                         fontsize=12, fontweight='bold', color='black')
    axes[0, 2].axis('off')

    # Get feature weights
    features = explanation.local_exp[pred_class_idx]
    feature_names = [f'Feature {f[0]}' for f in features]
    feature_weights = [f[1] for f in features]

    # Sort by absolute weight
    sorted_idx = np.argsort(np.abs(feature_weights))[::-1]
    top_features = sorted_idx[:10]

    # Plot feature importance
    y_pos = np.arange(len(top_features))
    axes[1, 0].barh(y_pos, [feature_weights[i] for i in top_features],
                    color=['green' if feature_weights[i] > 0 else 'red' for i in top_features])
    axes[1, 0].set_yticks(y_pos)
    axes[1, 0].set_yticklabels([f'F{i}' for i in top_features], fontsize=9)
    axes[1, 0].set_xlabel('Weight', fontsize=11, color='black')
    axes[1, 0].set_title('Top Feature Weights\n(Green: Positive, Red: Negative)',
                         fontsize=12, fontweight='bold', color='black')
    axes[1, 0].axvline(x=0, color='black', linestyle='-', linewidth=0.5)

    # Plot segmentation overlay
    axes[1, 1].imshow(img_display)

    # Get segmentation
    segmentation = explanation.segments

    # Overlay segmentation boundaries
    from skimage.segmentation import find_boundaries
    boundaries = find_boundaries(segmentation, mode='outer')
    axes[1, 1].imshow(np.ma.masked_where(~boundaries, boundaries),
                      cmap='gray', alpha=0.5)
    axes[1, 1].set_title('Image Segmentation\n(Superpixels)',
                         fontsize=12, fontweight='bold', color='black')
    axes[1, 1].axis('off')

    # Prediction probabilities
    preds = predict_fn(np.expand_dims(image, axis=0), model)[0]
    y_pos_prob = np.arange(len(class_names))
    colors = ['#4ECDC4' if i != pred_label else '#FF6B6B' for i in range(len(class_names))]
    axes[1, 2].barh(y_pos_prob, preds, color=colors)
    axes[1, 2].set_yticks(y_pos_prob)
    axes[1, 2].set_yticklabels(class_names, fontsize=9, color='black')
    axes[1, 2].set_xlim(0, 1)
    axes[1, 2].set_xlabel('Probability', fontsize=11, color='black')
    axes[1, 2].set_title('Prediction Probabilities',
                         fontsize=12, fontweight='bold', color='black')

    # Add confidence value
    axes[1, 2].text(0.5, -0.15, f'Confidence: {preds[pred_label]:.3f}',
                   transform=axes[1, 2].transAxes, ha='center',
                   fontsize=11, fontweight='bold', color='black')

    plt.suptitle(f'LIME Explanation - Sample {sample_idx}',
                 fontsize=14, fontweight='bold', color='black', y=1.02)
    plt.tight_layout()
    plt.show()

    # Print analysis
    print(f"\nLIME Analysis - Sample {sample_idx}:")
    print(f"  True class: {class_names[true_label]}")
    print(f"  Predicted class: {class_names[pred_label]}")
    print(f"  Confidence: {preds[pred_label]:.3f}")
    print(f"  Number of superpixels: {len(np.unique(explanation.segments))}")
    print("\n  Top influencing features:")
    for i, (feat_idx, weight) in enumerate(zip(top_features[:5], [feature_weights[i] for i in top_features[:5]])):
        direction = "SUPPORTS" if weight > 0 else "AGAINST"
        print(f"    {i+1}. Feature {feat_idx}: {weight:+.4f} ({direction})")
    print("-" * 60)

def run_lime_visualization(model, dataset, class_names, num_samples=3):
    """
    Run LIME visualization on sample images
    """
    print("\n" + "="*60)
    print("LIME EXPLANATION VISUALIZATION")
    print("="*60)

    # Setup LIME explainer
    explainer = setup_lime_explainer()
    print("✓ LIME explainer initialized")

    # Collect samples
    samples = []
    for images, labels in dataset.take(2):
        for i in range(min(3, len(images))):
            if len(samples) >= num_samples:
                break
            samples.append((images[i].numpy(), labels[i].numpy()))

    print(f"Processing {len(samples)} samples...\n")

    for idx, (img, true_label) in enumerate(samples):
        print(f"Generating LIME explanation for Sample {idx+1}...")

        # Prepare image (convert to 0-1 range for LIME)
        img_normalized = (img - img.min()) / (img.max() - img.min() + 1e-10)

        # Get prediction
        img_array = np.expand_dims(img, axis=0)
        preds = model.predict(img_array, verbose=0)[0]
        pred_label = np.argmax(preds)

        # Generate LIME explanation
        try:
            explanation = generate_lime_explanation(
                explainer,
                img_normalized,
                model,
                class_names,
                top_labels=3,
                num_samples=500  # Reduced for faster execution
            )

            # Visualize
            visualize_lime_explanation(
                img_normalized,
                explanation,
                true_label,
                pred_label,
                class_names,
                idx+1
            )

        except Exception as e:
            print(f"  Error generating LIME explanation: {e}")
            continue

    print("\n✓ LIME visualization complete!")

def run_lime_comparison(model, dataset, class_names, num_samples=2):
    """
    Compare LIME explanations for correct vs incorrect predictions
    """
    print("\n" + "="*60)
    print("LIME COMPARISON: CORRECT VS INCORRECT PREDICTIONS")
    print("="*60)

    explainer = setup_lime_explainer()

    # Collect correct and incorrect predictions
    correct_samples = []
    incorrect_samples = []

    for images, labels in dataset:
        preds = model.predict(images, verbose=0)
        pred_labels = np.argmax(preds, axis=1)

        for i in range(len(labels)):
            if len(correct_samples) >= num_samples and len(incorrect_samples) >= num_samples:
                break

            if labels[i] == pred_labels[i] and len(correct_samples) < num_samples:
                correct_samples.append((images[i].numpy(), labels[i].numpy(), pred_labels[i], preds[i]))
            elif labels[i] != pred_labels[i] and len(incorrect_samples) < num_samples:
                incorrect_samples.append((images[i].numpy(), labels[i].numpy(), pred_labels[i], preds[i]))

        if len(correct_samples) >= num_samples and len(incorrect_samples) >= num_samples:
            break

    # Show correct predictions
    if correct_samples:
        print("\n" + "-"*40)
        print("CORRECT PREDICTIONS")
        print("-"*40)

        for idx, (img, true_label, pred_label, probs) in enumerate(correct_samples):
            print(f"\nCorrect Prediction {idx+1}:")
            print(f"  True: {class_names[true_label]} → Pred: {class_names[pred_label]} (conf: {probs[pred_label]:.3f})")

            img_normalized = (img - img.min()) / (img.max() - img.min() + 1e-10)

            try:
                explanation = generate_lime_explanation(
                    explainer, img_normalized, model, class_names,
                    top_labels=3, num_samples=300
                )

                temp, mask = explanation.get_image_and_mask(
                    pred_label, positive_only=True, num_features=5, hide_rest=False
                )

                fig, axes = plt.subplots(1, 2, figsize=(10, 4))

                axes[0].imshow(img_normalized)
                axes[0].set_title(f'Original\nTrue: {class_names[true_label]}',
                                 fontsize=11, fontweight='bold', color='black')
                axes[0].axis('off')

                axes[1].imshow(mark_boundaries(temp, mask))
                axes[1].set_title(f'LIME Explanation\n(Important regions)',
                                 fontsize=11, fontweight='bold', color='black')
                axes[1].axis('off')

                plt.suptitle(f'Correct Prediction - {class_names[true_label]}',
                           fontsize=12, fontweight='bold', color='black')
                plt.tight_layout()
                plt.show()

            except Exception as e:
                print(f"  Error: {e}")

    # Show incorrect predictions
    if incorrect_samples:
        print("\n" + "-"*40)
        print("INCORRECT PREDICTIONS")
        print("-"*40)

        for idx, (img, true_label, pred_label, probs) in enumerate(incorrect_samples):
            print(f"\nIncorrect Prediction {idx+1}:")
            print(f"  True: {class_names[true_label]} → Pred: {class_names[pred_label]} (conf: {probs[pred_label]:.3f})")

            img_normalized = (img - img.min()) / (img.max() - img.min() + 1e-10)

            try:
                explanation = generate_lime_explanation(
                    explainer, img_normalized, model, class_names,
                    top_labels=3, num_samples=300
                )

                temp, mask = explanation.get_image_and_mask(
                    pred_label, positive_only=True, num_features=5, hide_rest=False
                )

                fig, axes = plt.subplots(1, 2, figsize=(10, 4))

                axes[0].imshow(img_normalized)
                axes[0].set_title(f'Original\nTrue: {class_names[true_label]}',
                                 fontsize=11, fontweight='bold', color='black')
                axes[0].axis('off')

                axes[1].imshow(mark_boundaries(temp, mask))
                axes[1].set_title(f'LIME Explanation\n(Why it thought {class_names[pred_label]})',
                                 fontsize=11, fontweight='bold', color='black')
                axes[1].axis('off')

                # Add red border for incorrect prediction
                for spine in axes[1].spines.values():
                    spine.set_edgecolor('red')
                    spine.set_linewidth(2)

                plt.suptitle(f'Incorrect Prediction - {class_names[true_label]} → {class_names[pred_label]}',
                           fontsize=12, fontweight='bold', color='black')
                plt.tight_layout()
                plt.show()

            except Exception as e:
                print(f"  Error: {e}")

def run_simple_lime(model, dataset, class_names, num_samples=2):
    """
    Simplified LIME visualization (faster and more reliable)
    """
    print("\n" + "="*60)
    print("SIMPLE LIME VISUALIZATION")
    print("="*60)

    explainer = setup_lime_explainer()

    samples = []
    for images, labels in dataset.take(2):
        for i in range(min(2, len(images))):
            if len(samples) >= num_samples:
                break
            samples.append((images[i].numpy(), labels[i].numpy()))

    for idx, (img, true_label) in enumerate(samples):
        print(f"\nProcessing Sample {idx+1}...")

        img_normalized = (img - img.min()) / (img.max() - img.min() + 1e-10)

        img_array = np.expand_dims(img, axis=0)
        preds = model.predict(img_array, verbose=0)[0]
        pred_label = np.argmax(preds)

        try:
            # Generate explanation with fewer samples for speed
            explanation = explainer.explain_instance(
                img_normalized,
                lambda x: predict_fn(x, model),
                top_labels=1,
                hide_color=0,
                num_samples=200,
                random_seed=42
            )

            # Get visualization
            temp, mask = explanation.get_image_and_mask(
                pred_label,
                positive_only=True,
                num_features=5,
                hide_rest=False
            )

            # Plot
            fig, axes = plt.subplots(1, 3, figsize=(15, 4))

            axes[0].imshow(img_normalized)
            axes[0].set_title(f'Original\nTrue: {class_names[true_label]}',
                             fontsize=11, fontweight='bold', color='black')
            axes[0].axis('off')

            axes[1].imshow(mark_boundaries(temp, mask))
            axes[1].set_title(f'LIME - Important Regions\nPred: {class_names[pred_label]}',
                             fontsize=11, fontweight='bold', color='black')
            axes[1].axis('off')

            # Confidence bar
            colors = ['#4ECDC4' if i != pred_label else '#FF6B6B' for i in range(len(class_names))]
            axes[2].barh(class_names, preds, color=colors)
            axes[2].set_xlim(0, 1)
            axes[2].set_xlabel('Confidence', fontsize=11, color='black')
            axes[2].set_title(f'Confidence: {preds[pred_label]:.3f}',
                             fontsize=11, fontweight='bold', color='black')

            plt.suptitle(f'LIME Explanation - Sample {idx+1}',
                        fontsize=13, fontweight='bold', color='black')
            plt.tight_layout()
            plt.show()

        except Exception as e:
            print(f"  Error: {e}")

# ============================================================
# ADD TO YOUR EXISTING MENU
# ============================================================

# Add this to your existing menu options
print("\nChoose visualization:")
print("1. Prediction Samples (with confidence bars)")
print("2. Confusion Matrix")
print("3. Misclassification Analysis")
print("4. Accuracy Summary")
print("5. LIME Visualization")
print("6. LIME Comparison (Correct vs Incorrect)")
print("7. Simple LIME (Fast)")

choice = input("\nEnter your choice (1-7): ")

if choice == '1':
    visualize_predictions(best_model, test_ds, class_names, num_samples=5)
elif choice == '2':
    show_confusion_matrix_with_predictions(best_model, test_ds, class_names)
elif choice == '3':
    show_misclassifications(best_model, test_ds, class_names, num_samples=3)
elif choice == '4':
    show_accuracy_summary(best_model, test_ds, class_names)
elif choice == '5':
    run_lime_visualization(best_model, test_ds, class_names, num_samples=2)
elif choice == '6':
    run_lime_comparison(best_model, test_ds, class_names, num_samples=1)
else:
    run_simple_lime(best_model, test_ds, class_names, num_samples=2)

print("\n✓ All visualizations complete!")